# TP-Q27-THROUGHPUT-PROBE v1.2.4 MASTER — comprehensive serving / cache / topology probe

**Goal.** Measure Qwen3.8-27B-FP8 serving on Kaggle 4×L4 without confusing engine throughput with ARC solve quality. This version is source-aligned to the audited PIV prompt/tool transport and packages enough raw evidence to diagnose hidden throughput, cache, scheduler, topology, thermal, and run-order effects.

### What v1.2.4 fixes (audit of v1.2.1)

- **PROBE_GPU crash fixed:** the topology pairing sweep used `{{ ... }}` (a set containing a dict → `TypeError`) and would have killed Cell 5 after the whole ladder, before analysis and bundle. The sweep code path is now identical in SELFTEST_CPU and PROBE_GPU, so the selftest exercises it.
- **Numerics gate is a threshold, not bit-exactness:** CUDA graphs/torch.compile fusions, custom all-reduce and TP2 partitioning legitimately change low-order bits, so requiring an exact 4×64 greedy match vs C0 would have excluded almost every real candidate and reported the frozen engine as the "winner". Candidates now require `mean_prefix_agreement ≥ NUMERICS_MIN_PREFIX_AGREEMENT` (0.50); exactness is still reported. TP2×2 pairs require their single-TP2 row to be `ok`, not bit-exact.
- **Timeouts/projection:** per-config timeouts raised (SMOKE 60 min, PROBE 75 min); the deadline projection now includes the long anchor and the prefill waves; SMOKE budget 4.5 h.
- **Prefill waves only where they carry information:** `tool_history_seed` and the three extension waves run only on rows with `enable_prefix_caching=True` (elsewhere they measure nothing but cost ~10 min per row). Cold + exact-repeat still run everywhere.
- **Drift sentinel truly last:** `Z0_frozen_q1_repeat` now runs after the TP2×2 pair tests.
- Replica prompt offset is honoured (the two TP2 replicas take disjoint prompt slices); leftover GPU processes that survived a reclaim are never re-labelled as "baseline".

### What v1.2.1 added beyond v1.2

- **PIV-realistic prompt transport:** initial prompts use the audited PiVOTools-aligned ARC prompt, Qwen native Python tool schema, `enable_thinking=True`, `reasoning_effort="xhigh"`, and the mounted Qwen chat template. No plain-text fallback on GPU.
- **Realistic multi-turn cache workload:** tool-history seed + tool-response next-turn prompts are rendered through the chat template. Small/medium/large tool-result payloads and a 256-token next-agent-turn decode are measured. Shared-prefix token counts are computed from token IDs, not assumed.
- **Direct cache evidence:** vLLM `RequestOutput.num_cached_tokens` is captured per request and compared with the expected shared prefix.
- **Request-level latency evidence:** first-token latency, queue interval, decode interval, per-request decode tok/s, and latency tails are retained when vLLM exposes them.
- **Session drift sentinel:** the frozen Q1 engine is reloaded at the end and `decode@8` is repeated. A large start→end drift invalidates cross-config ranking.
- **Topology sweep:** SMOKE measures AR/no-AR on the default TP2×2 pairing. Full PROBE also measures the other two 4-GPU perfect matchings for the better AR mode.
- **Hardware evidence:** 2-second GPU telemetry (memory/utilization/power/temperature/clocks/pstate), driver/topology snapshots, package/runtime evidence, preemption/error lines, energy estimates, and cleanup state.
- **Master evidence contract:** flat CSV/JSON analysis files, raw specs/results/logs, prompt batch plan, environment snapshots, `SELFCHECK.json`, SHA256 manifest, and `RETURN_TO_CHATGPT.json` are all placed in one runtime evidence ZIP.

### Profiles

- `SELFTEST_CPU` — standalone fake engine; no Kaggle inputs required. Exercises crash, hang, deadline skip, cache-hit metrics, request metrics, drift sentinel, pair orchestration, analysis, manifest, and ZIP.
- `SMOKE_GPU` — **default**. Anchor + graph/AR + TP2 feasibility + default TP2×2 pair modes + end drift sentinel. Run this before the full probe.
- `PROBE_GPU` — full branch matrix: batch/scheduler/cache/FP8-KV/memory/experimental-MTP rows, topology pairing sweep, richer stability and multi-turn measurements.

### Interpretation guardrails

This notebook never auto-promotes a config into the ARC solver. The C0 anchor and end-of-run drift gate must pass before speed comparisons are trusted. Greedy fingerprint drift, cache-hit evidence, stability, preemption, thermal/power state, and real solver follow-up remain separate from raw tok/s. The 300 tok/s target is an engineering target, **not** inferred from leaderboard history.


In [1]:
# =====================================================================================
# CELL 1 — CONFIG: TP-Q27-THROUGHPUT-PROBE v1.2.4 MASTER
# Engine-throughput ladder for Qwen3.8-27B-FP8 on Kaggle 4xL4 (vLLM 0.19.0, TP=4).
# Each engine configuration runs in its OWN subprocess (fresh CUDA state); a failed or
# hung configuration is recorded and the ladder continues. Nothing here changes what the
# model is asked to do -- only how fast the engine serves it.
# =====================================================================================
import os, sys, json, time, math, hashlib, shutil, subprocess, importlib.metadata, platform, signal, re
from pathlib import Path
from datetime import datetime, timezone

EXPERIMENT_ID = "TP-Q27-THROUGHPUT-PROBE-v1.2.4-MASTER"
RUN_PROFILE = "SMOKE_GPU"            # "SELFTEST_CPU" | "SMOKE_GPU" | "PROBE_GPU"
SEED = 42

# Optional: restrict the ladder to a subset of config names (None = profile default).
PROBE_CONFIGS = None                 # e.g. ["C0_frozen_q1", "C1_graph_noAR"]

PROFILE_TABLE = {
    "SELFTEST_CPU": {"use_model": False, "fake_engine": True, "output_tokens": 64, "prompt_pool": 64,
                     "config_filter": ["C0_frozen_q1", "I0_q1_stats", "C1_graph_noAR", "C3_noAR_mns64", "S1_noAR_mns64_mbt64k", "C5_noAR_prefix64", "X1_noAR_mns64_mtp1", "T1_tp2_noAR", "T2_tp2_AR", "Z0_frozen_q1_repeat"], "per_config_timeout_sec": 120, "total_budget_sec": 5 * 60 * 60,
                     "stability_tokens": 32, "stability_repeats": 3, "mixed_batch_n": 16,
                     "bundle_name": "TP_Q27_PROBE_V124_SELFTEST_MASTER_EVIDENCE.zip"},
    "SMOKE_GPU":    {"use_model": True, "fake_engine": False, "output_tokens": 256, "prompt_pool": 96,
                     "config_filter": ["C0_frozen_q1", "I0_q1_stats", "C1_graph_noAR", "C2_graph_AR", "C5_noAR_prefix64", "T1_tp2_noAR", "T2_tp2_AR", "Z0_frozen_q1_repeat"],
                     "per_config_timeout_sec": 75 * 60, "total_budget_sec": 6 * 60 * 60,
                     "stability_tokens": 128, "stability_repeats": 2, "mixed_batch_n": 16,
                     "bundle_name": "TP_Q27_PROBE_V124_SMOKE_MASTER_EVIDENCE.zip"},
    "PROBE_GPU":    {"use_model": True, "fake_engine": False, "output_tokens": 1024, "prompt_pool": 120,
                     "config_filter": None, "per_config_timeout_sec": 75 * 60, "total_budget_sec": 11 * 60 * 60,
                     "stability_tokens": 256, "stability_repeats": 2, "mixed_batch_n": 32,
                     "bundle_name": "TP_Q27_PROBE_V124_GPU_MASTER_EVIDENCE.zip"},
}
if RUN_PROFILE not in PROFILE_TABLE:
    raise RuntimeError(f"Unknown RUN_PROFILE {RUN_PROFILE!r}; choose one of {list(PROFILE_TABLE)}")
PROFILE = PROFILE_TABLE[RUN_PROFILE]
USE_MODEL = bool(PROFILE["use_model"])
FAKE_ENGINE = bool(PROFILE["fake_engine"])
OUTPUT_TOKENS = int(PROFILE["output_tokens"])
PROMPT_POOL_SIZE = int(PROFILE["prompt_pool"])
PER_CONFIG_TIMEOUT_SEC = float(PROFILE["per_config_timeout_sec"])
TOTAL_BUDGET_SEC = float(PROFILE["total_budget_sec"])
STABILITY_TOKENS = int(PROFILE["stability_tokens"])
STABILITY_REPEATS = int(PROFILE["stability_repeats"])
MIXED_BATCH_N = int(PROFILE["mixed_batch_n"])
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
BUNDLE_NAME = str(PROFILE["bundle_name"]).replace(".zip", f"_{RUN_ID}.zip")

# ---- Kaggle inputs (identical to Q1 / PIV-Q27-ACT v1.9). TP_PROBE_INPUT_ROOT lets the CPU
# ---- selftest point at a local stand-in tree; on Kaggle leave it unset.
INPUT_ROOT = Path(os.environ.get("TP_PROBE_INPUT_ROOT", "/kaggle/input"))
MODEL_PATH = INPUT_ROOT / "models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"
WHEELHOUSE = INPUT_ROOT / "datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
ARC_ROOT = INPUT_ROOT / "competitions/arc-prize-2026-arc-agi-2"
EVAL_CHALLENGES_PATH = ARC_ROOT / "arc-agi_evaluation_challenges.json"
WORKING_ROOT = Path(os.environ.get("TP_PROBE_WORKING_ROOT", "/kaggle/working"))
# Run-ID-scoped directory: rerunning Cell 1 cannot recursively delete evidence from a prior run.
WORK_ROOT = WORKING_ROOT / "tp_q27_probe_v124" / RUN_PROFILE.lower() / RUN_ID
for _sub in ("specs", "results", "logs", "gpu_samples", "analysis", "environment", "measurement"):
    (WORK_ROOT / _sub).mkdir(parents=True, exist_ok=False)
NOTEBOOK_WALL_START = time.perf_counter()
DEADLINE = NOTEBOOK_WALL_START + TOTAL_BUDGET_SEC

# ---- Q1-frozen engine (byte-for-byte the ENGINE_CONFIG of Q1 / PIV v1.7-v1.9, minus the model path).
Q1_FROZEN_ENGINE = {
    "tensor_parallel_size": 4,
    "distributed_executor_backend": "mp",
    "dtype": "auto",
    "load_format": "safetensors",
    "trust_remote_code": False,
    "language_model_only": True,
    "max_model_len": 29696,
    "max_num_seqs": 16,
    "max_num_batched_tokens": 32768,
    "gpu_memory_utilization": 0.90,
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "enable_prefix_caching": False,
    "seed": SEED,
}
MAX_MODEL_LEN = int(Q1_FROZEN_ENGINE["max_model_len"])


# ---- Controlled branch matrix. No later row silently inherits custom all-reduce.
# ---- visible_devices allows TP=2 feasibility rows to use only one L4 pair.
_INSTRUMENT = {"disable_log_stats": False}
_GRAPH_NOAR = {"enforce_eager": False, "disable_custom_all_reduce": True, "disable_log_stats": False}
_GRAPH_AR   = {"enforce_eager": False, "disable_custom_all_reduce": False, "disable_log_stats": False}
CONFIG_LADDER = [
    {"name": "C0_frozen_q1", "overrides": {}, "decode_points": [8, 16], "visible_devices": "0,1,2,3", "anchor_long": True, "source_frozen": True,
     "hypothesis": "Frozen Q1 anchor. Dedicated long decode@8 must reproduce the prior ~0.16 s/token/sequence within tolerance; request stats remain source-faithful/off."},
    {"name": "I0_q1_stats", "overrides": dict(_INSTRUMENT), "decode_points": [8, 16], "visible_devices": "0,1,2,3",
     "hypothesis": "Instrumentation control: frozen Q1 engine + only disable_log_stats=False. Optimization rows compare against this when request-level latency is needed."},
    {"name": "C1_graph_noAR", "overrides": dict(_GRAPH_NOAR), "decode_points": [8, 16], "visible_devices": "0,1,2,3", "solver_long_n": 8,
     "hypothesis": "CUDA graphs/compile only; custom all-reduce remains disabled."},
    {"name": "C2_graph_AR", "overrides": dict(_GRAPH_AR), "decode_points": [8, 16], "visible_devices": "0,1,2,3", "solver_long_n": 8,
     "hypothesis": "Same as C1 plus custom all-reduce; isolates AR at TP4."},
    {"name": "C3_noAR_mns64", "overrides": dict(_GRAPH_NOAR, max_num_seqs=64), "decode_points": [16, 32, 64], "visible_devices": "0,1,2,3", "solver_long_n": 16, "agent_turn_n": 16,
     "hypothesis": "Batch scaling to 64 with custom all-reduce disabled."},
    {"name": "C3_AR_mns64", "overrides": dict(_GRAPH_AR, max_num_seqs=64), "decode_points": [16, 32, 64], "visible_devices": "0,1,2,3", "solver_long_n": 16, "agent_turn_n": 16,
     "hypothesis": "Batch scaling to 64 with custom all-reduce enabled; paired with C3_noAR_mns64."},
    {"name": "C4_noAR_mns128", "overrides": dict(_GRAPH_NOAR, max_num_seqs=128), "decode_points": [32, 64, 128], "visible_devices": "0,1,2,3",
     "hypothesis": "Upper batch scaling without custom all-reduce."},
    {"name": "C4_AR_mns128", "overrides": dict(_GRAPH_AR, max_num_seqs=128), "decode_points": [32, 64, 128], "visible_devices": "0,1,2,3",
     "hypothesis": "Upper batch scaling with custom all-reduce; paired with C4_noAR_mns128."},
    {"name": "S1_noAR_mns64_mbt64k", "overrides": dict(_GRAPH_NOAR, max_num_seqs=64, max_num_batched_tokens=65536), "decode_points": [16, 32, 64], "visible_devices": "0,1,2,3", "optional": True,
     "hypothesis": "Raise scheduler token budget from frozen 32k to 64k; isolates max_num_batched_tokens on graph/noAR."},
    {"name": "S2_noAR_mns64_mbt128k", "overrides": dict(_GRAPH_NOAR, max_num_seqs=64, max_num_batched_tokens=131072), "decode_points": [16, 32, 64], "visible_devices": "0,1,2,3", "optional": True,
     "hypothesis": "Further scheduler-token-budget sweep to 128k; may improve long-prompt prefill throughput or increase memory pressure."},
    {"name": "S3_noAR_mns64_async", "overrides": dict(_GRAPH_NOAR, max_num_seqs=64, async_scheduling=True), "decode_points": [16, 32, 64], "visible_devices": "0,1,2,3", "optional": True,
     "hypothesis": "vLLM async scheduling can reduce host-side scheduling gaps; record unsupported/conflict cleanly."},
    {"name": "S4_noAR_mns64_dbo", "overrides": dict(_GRAPH_NOAR, max_num_seqs=64, enable_dbo=True), "decode_points": [16, 32, 64], "visible_devices": "0,1,2,3", "optional": True,
     "hypothesis": "Dual-batch overlap probe; throughput evidence only and unsupported/conflicting configs are recorded."},
    {"name": "C5_noAR_prefix64", "overrides": dict(_GRAPH_NOAR, max_num_seqs=64, enable_prefix_caching=True), "decode_points": [32], "visible_devices": "0,1,2,3", "agent_turn_n": 16,
     "hypothesis": "Prefix caching with no custom AR; measure exact repeat and shared-prefix extension."},
    {"name": "C5_AR_prefix64", "overrides": dict(_GRAPH_AR, max_num_seqs=64, enable_prefix_caching=True), "decode_points": [32], "visible_devices": "0,1,2,3", "agent_turn_n": 16,
     "hypothesis": "Prefix caching with custom AR; paired with C5_noAR_prefix64."},
    {"name": "C6_noAR_prefix_align", "overrides": dict(_GRAPH_NOAR, max_num_seqs=64, enable_prefix_caching=True, mamba_cache_mode="align"), "decode_points": [32], "visible_devices": "0,1,2,3", "optional": True,
     "hypothesis": "Explicit hybrid-cache align mode without custom AR."},
    {"name": "C6_AR_prefix_align", "overrides": dict(_GRAPH_AR, max_num_seqs=64, enable_prefix_caching=True, mamba_cache_mode="align"), "decode_points": [32], "visible_devices": "0,1,2,3", "optional": True,
     "hypothesis": "Explicit hybrid-cache align mode with custom AR."},
    {"name": "C7_noAR_kvfp8", "overrides": dict(_GRAPH_NOAR, max_num_seqs=128, kv_cache_dtype="fp8"), "decode_points": [32, 64, 128], "visible_devices": "0,1,2,3", "optional": True,
     "hypothesis": "FP8 attention KV cache without custom AR. Capacity gain is measured, not assumed to be exactly 2x."},
    {"name": "C7_AR_kvfp8", "overrides": dict(_GRAPH_AR, max_num_seqs=128, kv_cache_dtype="fp8"), "decode_points": [32, 64, 128], "visible_devices": "0,1,2,3", "optional": True,
     "hypothesis": "FP8 attention KV cache with custom AR."},
    {"name": "C8_noAR_mns128_gpu95", "overrides": dict(_GRAPH_NOAR, max_num_seqs=128, gpu_memory_utilization=0.95), "decode_points": [32, 64, 128], "visible_devices": "0,1,2,3", "optional": True,
     "hypothesis": "Capacity-only memory-utilization sweep from 0.90 to 0.95; isolate extra KV headroom versus OOM risk."},
    {"name": "X1_noAR_mns64_mtp1", "overrides": dict(_GRAPH_NOAR, max_num_seqs=64, speculative_config={"method":"mtp","num_speculative_tokens":1}), "decode_points": [16, 32, 64], "visible_devices": "0,1,2,3", "optional": True, "experimental_no_solver_promotion": True,
     "hypothesis": "Native MTP-1 speculative decode throughput probe. Experimental only: vLLM 0.19 thinking-budget interactions require a later solver smoke."},
    {"name": "T1_tp2_noAR", "overrides": dict(_GRAPH_NOAR, tensor_parallel_size=2, max_num_seqs=64), "decode_points": [16, 32, 64], "visible_devices": "0,1", "solver_long_n": 16,
     "hypothesis": "Single TP2 replica on two L4s, custom AR disabled. Tests whether the 27B FP8 checkpoint fits and scales."},
    {"name": "T2_tp2_AR", "overrides": dict(_GRAPH_AR, tensor_parallel_size=2, max_num_seqs=64), "decode_points": [16, 32, 64], "visible_devices": "0,1", "solver_long_n": 16,
     "hypothesis": "Single TP2 replica on two L4s, custom AR enabled. Paired with T1."},
    {"name": "Z0_frozen_q1_repeat", "overrides": {}, "decode_points": [8], "visible_devices": "0,1,2,3", "drift_sentinel": True, "source_frozen": True,
     "hypothesis": "End-of-run frozen-Q1 sentinel. Repeats the same decode@8 workload to measure session/order/thermal drift."},
]
if PROFILE["config_filter"]:
    CONFIG_LADDER = [c for c in CONFIG_LADDER if c["name"] in PROFILE["config_filter"]]
if PROBE_CONFIGS:
    _unknown = [n for n in PROBE_CONFIGS if n not in {c["name"] for c in CONFIG_LADDER}]
    if _unknown:
        raise RuntimeError(f"PROBE_CONFIGS names not in ladder: {_unknown}")
    CONFIG_LADDER = [c for c in CONFIG_LADDER if c["name"] in PROBE_CONFIGS]
if FAKE_ENGINE:
    # Synthetic rows that MUST fail / hang / be skipped, to prove those paths end in a bundle, not a crash.
    CONFIG_LADDER = CONFIG_LADDER + [
        {"name": "ZZ_fake_fail", "overrides": dict(_GRAPH_NOAR), "decode_points": [8], "hypothesis": "selftest: engine init raises"},
        {"name": "ZZ_fake_hang", "overrides": dict(_GRAPH_NOAR), "decode_points": [8], "hypothesis": "selftest: worker hangs -> parent timeout kill", "timeout_sec": 3},
        {"name": "ZZ_fake_skip", "overrides": dict(_GRAPH_NOAR), "decode_points": [8], "hypothesis": "selftest: projection exceeds budget -> skipped_deadline", "projected_sec_override": 1e9, "optional": True},
    ]
if not CONFIG_LADDER:
    raise RuntimeError("Empty CONFIG_LADDER after filtering.")

# ---- Measurement constants
PREFILL_PROMPTS = 32                 # prompts used for the prefill cold/warm measurement
FINGERPRINT_PROMPTS = 4              # greedy 64-token fingerprint for numerics comparison
FINGERPRINT_TOKENS = 64
WARMUP_PROMPTS = 2
GPU_SAMPLE_INTERVAL_SEC = 2.0        # nvidia-smi sampling while a config runs
PRIOR_LOAD_SEC = 420.0               # projection prior before the first measured load (Q1: 277 s + compile margin)
PRIOR_SEC_PER_OUTPUT_TOKEN = 0.16    # Q1/PIV measured per-sequence step time
CONFIG_OVERHEAD_SEC = 240.0          # warmup + prefill + fingerprint + teardown allowance
THROUGHPUT_TARGET_TPS = 300.0        # engineering goal only; NOT inferred from leaderboard scores
THROUGHPUT_TARGET_PROVENANCE = "engineering_goal_not_leaderboard_inference"
ANCHOR_OUTPUT_TOKENS = 4096             # long enough that prefill is a small fraction of Q1-style timing
ANCHOR_BATCH_N = 8
ANCHOR_EXPECTED_SEC_PER_TOKEN = 0.16
ANCHOR_RATIO_BOUNDS = (0.60, 1.60)
SOLVER_LONG_OUTPUT_TOKENS = 3072       # matches the active PIV reasoning-budget scale; measured on selected configs
AGENT_TURN_OUTPUT_TOKENS = 256         # cached tool-history -> next-assistant decode representative
CACHE_HIT_VERIFY_MIN_RATIO = 0.50      # conservative due block boundaries / hybrid cache granularity
SESSION_DRIFT_RATIO_BOUNDS = (0.80, 1.20)
PREFIX_EXTENSION_SMALL_TEXT = "\n\nTool result: inspected puzzle metadata and candidate objects. Continue from the cached conversation prefix."
_PREFIX_TOOL_ROW = "\nobject: color=3 bbox=(2,3,5,7) size=12 relation=inside candidate=rot90 score=0.75"
PREFIX_EXTENSION_TEXT = "\n\nTool result (medium representative payload):" + (_PREFIX_TOOL_ROW * 32)
PREFIX_EXTENSION_LARGE_TEXT = "\n\nTool result (large representative payload):" + (_PREFIX_TOOL_ROW * 128)
PREFIX_AGENT_TURN_TEXT = "Tool result (next-turn decode payload):" + (_PREFIX_TOOL_ROW * 20)
TP2_PAIR_N = 32                         # per replica => 64 concurrent sequences across 4 GPUs
TP2_PAIR_TIMEOUT_SEC = 55 * 60
TP2_PAIR_READY_MAX_SEC = 25 * 60
CACHE_AB_REPEATS = 3
NUMERICS_REQUIRE_EXACT_GREEDY = False   # v1.2.4: exactness is reported, not required
NUMERICS_MIN_PREFIX_AGREEMENT = 0.50    # candidate gate: mean greedy prefix agreement vs C0 (4 prompts x 64 tokens)
PRIOR_PREFILL_TPS = 1500.0              # projection prior for prefill waves before the first measured cold prefill
BALANCE_BUCKETS = 8
MIXED_OUTPUT_FRACTIONS = (0.25, 0.50, 0.75, 1.00)
GPU_TELEMETRY_FIELDS = ("index", "memory.used", "utilization.gpu", "utilization.memory", "temperature.gpu", "power.draw", "pstate", "clocks.sm", "clocks.mem")
PARENT_SOURCE_FINGERPRINT_ALGO = "sha256(code_cells; normalize fingerprint assignment + RUN_PROFILE value)"
PARENT_SOURCE_FINGERPRINT_SHA256 = "106e6443a461f282bed45bb36eba80da7989feb2add0af510f16062de2ae795c"
MEASUREMENT_CONTRACT = {
    "decode": ["balanced_prompt_lengths", "fixed_length_capacity", "mixed_output_lengths", "stability_cv", "long_c0_anchor", "solver_representative_3072_token_decode", "request_level_ttft_decode_tail", "end_session_drift_sentinel"],
    "prefill": ["cache_reset", "cold", "exact_repeat", "matched_uncached_vs_cached_extension_repeats", "tool_history_seed", "cached_tool_history_extension_small", "cached_tool_history_extension_medium", "cached_tool_history_extension_large", "cached_next_agent_turn_decode", "vllm_num_cached_tokens", "expected_shared_prefix_tokens", "new_suffix_tokens"],
    "engine": ["eager_vs_graph", "custom_all_reduce", "max_num_seqs", "max_num_batched_tokens", "async_scheduling", "dbo", "prefix_cache", "mamba_cache_mode", "fp8_kv", "gpu_memory_utilization", "mtp1_experimental"],
    "topology": ["tp4", "tp2_single", "tp2x2_noAR", "tp2x2_AR", "tp2x2_alternative_pairings_in_full_probe"],
    "hardware": ["topology", "memory", "utilization", "power", "temperature", "clocks", "pstate", "energy_per_token"],
    "runtime_health": ["request_statistics_enabled", "preemption_log_lines", "oom_error_hints", "teardown_reclaim", "dirty_gpu_fail_closed", "timeout", "unsupported_kwargs", "request_timing_metrics", "phase_output_integrity", "session_drift"],
    "reproducibility": ["runtime_stack", "driver_topology", "cpu_ram_disk", "pip_freeze", "prompt_batch_plan", "tp2_pair_prompt_plan", "parent_source_fingerprint", "manifest_sha256"],
    "safety": ["greedy_fingerprint_threshold", "anchor_gate", "unsupported_kwargs", "oom_timeout_cleanup", "manifest_sha256"],
}

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  # parent owns no CUDA context; child rows may override
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

def pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def stable_json_sha256(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def small_file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=str, ensure_ascii=False)

def elapsed_sec():
    return time.perf_counter() - NOTEBOOK_WALL_START

if USE_MODEL:
    required = [MODEL_PATH, ARC_ROOT, EVAL_CHALLENGES_PATH, WHEELHOUSE]
    for p in required:
        if not p.exists():
            raise FileNotFoundError(f"Required input is missing: {p}")
    config_path = MODEL_PATH / "config.json"
    index_path = MODEL_PATH / "model.safetensors.index.json"
    with open(config_path, "r", encoding="utf-8") as f:
        model_cfg = json.load(f)
    qcfg = model_cfg.get("quantization_config", {})
    if model_cfg.get("model_type") != "qwen3_5":
        raise RuntimeError(f"Unexpected model_type={model_cfg.get('model_type')}")
    if qcfg.get("quant_method") != "fp8" or qcfg.get("fmt") != "e4m3":
        raise RuntimeError(f"Unexpected quantization_config={qcfg}")
    MODEL_ARTIFACT_FINGERPRINT_PAYLOAD = {
        "model_path": str(MODEL_PATH),
        "config_sha256": small_file_sha256(config_path),
        "safetensors_index_sha256": small_file_sha256(index_path) if index_path.exists() else None,
    }
else:
    MODEL_ARTIFACT_FINGERPRINT_PAYLOAD = {"mode":"SELFTEST_CPU", "model_path":None, "synthetic":True}
MODEL_ARTIFACT_FINGERPRINT_SHA256 = stable_json_sha256(MODEL_ARTIFACT_FINGERPRINT_PAYLOAD)
write_json(WORK_ROOT / "measurement" / "MEASUREMENT_CONTRACT.json", MEASUREMENT_CONTRACT)

print("=" * 116)
print(EXPERIMENT_ID, "| engine-throughput ladder, one subprocess per engine config")
print("=" * 116)
print("RUN_ID:", RUN_ID, "| RUN_PROFILE:", RUN_PROFILE, "| fake_engine:", FAKE_ENGINE)
print("OUTPUT_TOKENS per sequence:", OUTPUT_TOKENS, "| prompt pool:", PROMPT_POOL_SIZE, "| budget:", TOTAL_BUDGET_SEC / 3600, "h")
print("Ladder:", [c["name"] for c in CONFIG_LADDER])
print("Q1 frozen engine sha256:", stable_json_sha256(Q1_FROZEN_ENGINE))
print("Model artifact fingerprint:", MODEL_ARTIFACT_FINGERPRINT_SHA256)
print("WORK_ROOT:", WORK_ROOT)


TP-Q27-THROUGHPUT-PROBE-v1.2.4-MASTER | engine-throughput ladder, one subprocess per engine config
RUN_ID: 20260916T091344Z | RUN_PROFILE: SMOKE_GPU | fake_engine: False
OUTPUT_TOKENS per sequence: 256 | prompt pool: 96 | budget: 6.0 h
Ladder: ['C0_frozen_q1', 'I0_q1_stats', 'C1_graph_noAR', 'C2_graph_AR', 'C5_noAR_prefix64', 'T1_tp2_noAR', 'T2_tp2_AR', 'Z0_frozen_q1_repeat']
Q1 frozen engine sha256: 38daeec3e924f73faed3e64f347e46d607e0479c9aaa95bc7298093a59e6ea64
Model artifact fingerprint: 9a3766b0a840b48e9a692d3bb8bc0504b990e125cccf8c30f6817b7b1c6d3972
WORK_ROOT: /kaggle/working/tp_q27_probe_v124/smoke_gpu/20260916T091344Z


In [2]:
# =====================================================================================
# CELL 2 — RUNTIME PARITY + ENVIRONMENT / TOPOLOGY EVIDENCE
# Parent process never imports torch/vLLM. GPU profile pins the Q1/PIV stack.
# =====================================================================================
EXPECTED_RUNTIME_PACKAGES = {"vllm": "0.19.0", "transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}

def offline_pip_install(requirements):
    if not requirements:
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])

before_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
if USE_MODEL:
    for _pkg, _module in {"transformers": "transformers", "tokenizers": "tokenizers", "huggingface-hub": "huggingface_hub"}.items():
        if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _module in sys.modules:
            raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart the Kaggle session before Run All.")
    if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        offline_pip_install(["vllm==0.19.0"])
    else:
        repair = [f"{p}=={v}" for p, v in EXPECTED_RUNTIME_PACKAGES.items() if p != "vllm" and before_stack.get(p) != v]
        offline_pip_install(repair)
    after_stack = {name: pkg_version(name) for name in EXPECTED_RUNTIME_PACKAGES}
    for package, wanted in EXPECTED_RUNTIME_PACKAGES.items():
        if after_stack.get(package) != wanted:
            raise RuntimeError(f"Runtime parity failure: {package}={after_stack.get(package)!r}, expected {wanted!r}")
    print("Runtime dependency parity PASS:", after_stack)
else:
    after_stack = dict(before_stack)
    print("SELFTEST_CPU: no vLLM/model dependency required; runtime stack as found:", after_stack)

RUNTIME_STACK = dict(after_stack)
RUNTIME_STACK["python"] = platform.python_version()
RUNTIME_STACK_SHA256 = stable_json_sha256(RUNTIME_STACK)

def _run_text(cmd, timeout=30):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return {"cmd": cmd, "returncode": p.returncode, "stdout": p.stdout, "stderr": p.stderr}
    except Exception as e:
        return {"cmd": cmd, "returncode": None, "stdout": "", "stderr": f"{type(e).__name__}: {e}"}

def nvidia_smi_query(fields, extra=()):
    """Return list of rows or None when nvidia-smi/query is unavailable."""
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={','.join(fields)}", "--format=csv,noheader,nounits", *extra],
                             capture_output=True, text=True, timeout=20)
    except (OSError, subprocess.TimeoutExpired):
        return None
    if out.returncode != 0:
        return None
    return [[c.strip() for c in line.split(",")] for line in out.stdout.strip().splitlines() if line.strip()]

GPU_INVENTORY = nvidia_smi_query(["index", "name", "uuid", "pci.bus_id", "memory.total", "memory.used", "driver_version"]) or []
print("GPU inventory (nvidia-smi):", GPU_INVENTORY if GPU_INVENTORY else "unavailable")
if USE_MODEL and len(GPU_INVENTORY) != 4:
    raise RuntimeError(f"Expected 4 GPUs via nvidia-smi, saw {len(GPU_INVENTORY)}. Select the Kaggle 4xL4 accelerator.")

ENV_DIR = WORK_ROOT / "environment"
ENV_DIR.mkdir(parents=True, exist_ok=True)
for name, cmd in {
    "nvidia_smi_L.txt": ["nvidia-smi", "-L"],
    "nvidia_smi_topo_m.txt": ["nvidia-smi", "topo", "-m"],
    "nvidia_smi_q.txt": ["nvidia-smi", "-q"],
    "uname.txt": ["uname", "-a"],
    "lscpu.txt": ["lscpu"],
    "memory.txt": ["bash", "-lc", "free -h; echo; cat /proc/meminfo | head -40"],
    "disk.txt": ["df", "-h"],
    "ulimit.txt": ["bash", "-lc", "ulimit -a"],
    "pip_freeze.txt": [sys.executable, "-m", "pip", "freeze"],
}.items():
    rec = _run_text(cmd, timeout=60)
    (ENV_DIR / name).write_text(rec["stdout"] + ("\nSTDERR:\n" + rec["stderr"] if rec["stderr"] else ""), encoding="utf-8")
write_json(ENV_DIR / "runtime_stack.json", RUNTIME_STACK)
write_json(ENV_DIR / "gpu_inventory.json", GPU_INVENTORY)
write_json(ENV_DIR / "selected_environment.json", {k: os.environ.get(k) for k in ["CUDA_VISIBLE_DEVICES","CUDA_DEVICE_ORDER","VLLM_WORKER_MULTIPROC_METHOD","HF_HUB_OFFLINE","TRANSFORMERS_OFFLINE"]})
write_json(ENV_DIR / "host_runtime.json", {"platform": platform.platform(), "python": platform.python_version(), "cpu_count": os.cpu_count(), "machine": platform.machine(), "processor": platform.processor()})


Looking in links: /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/vllm-0.19.0-cp38-abi3-manylinux_2_31_x86_64.whl
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/transformers-4.57.6-py3-none-any.whl (from vllm==0.19.0)
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl (from vllm==0.19.0)
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/prometheus_fastapi_instrumentator-7.1.0-py3-none-any.whl (from vllm==0.19.0)
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/lm_format_enforcer-0.11.3-py3-none-any.whl (from vllm==0.19.0)
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3/llguidance-1.3.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from vllm==0.19.0)
Processing /kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheel

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.


Runtime dependency parity PASS: {'vllm': '0.19.0', 'transformers': '4.57.6', 'tokenizers': '0.22.2', 'huggingface-hub': '0.36.2'}
GPU inventory (nvidia-smi): [['0', 'NVIDIA L4', 'GPU-b3cc3d80-2350-cd08-8847-360a088282c5', '00000000:00:03.0', '23034', '0', '580.159.04'], ['1', 'NVIDIA L4', 'GPU-4d032e80-5daf-1e7a-4221-0f509b3873ff', '00000000:00:04.0', '23034', '0', '580.159.04'], ['2', 'NVIDIA L4', 'GPU-b60e9ad0-8b6d-63d9-a51e-d813fd8f5f81', '00000000:00:05.0', '23034', '0', '580.159.04'], ['3', 'NVIDIA L4', 'GPU-d1f86170-bd5c-7da8-b418-ce28254e32ef', '00000000:00:06.0', '23034', '0', '580.159.04']]


In [3]:
# =====================================================================================
# CELL 3 — PIV-SOURCE-ALIGNED PROMPT POOL + TOOL-HISTORY CACHE WORKLOAD + BALANCED BATCH PLAN
# Real GPU: audited PiVOTools-aligned ARC prompt + Qwen Python tool transport; fail closed.
# SELFTEST_CPU: deterministic synthetic prompts; no Kaggle/model files required.
# =====================================================================================
import numpy as np

PIV_SOURCE_SYSTEM_PROMPT = '# Tools\n\n## python\n\nUse this tool to execute Python code in your chain of thought. \n\nWhen you send a message containing Python code to python, it will be executed in a stateful Jupyter notebook environment. The environment supports common third party libraries like numpy, scipy, shapely, networkx, scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver. python will respond with the output of the execution or time out after 120.0 seconds. \n\n### Usage guideline\n\n1. Use the variables and functions already loaded in the Jupyter notebook instead of defining them yourself. The user message will provide a complete list of such variables and functions.\n2. Prefer to write reusable functions. These will persist in the stateful Jupyter notebook and can be referenced in later code snippets without having to define them again.'
PIV_SOURCE_AGENTIC_PROMPT_TEMPLATE = 'You are an expert ARC AGI puzzle solver. Please solve the puzzle given below.\n\n# Background information about ARC AGI puzzles\n\nEach task consists of a small number of demonstration examples (3.3 on average), and a small number of test examples (1 - 3).\n\nEach example consists of an “input grid” and an “output grid”. Each “grid” is a literal grid of colored squares There are 10 unique colors. A grid can be any height or width between 1x1 and 30x30, inclusive.\n\nThe color code is as follows:\n\n0 - black\n1 - navy blue\n2 - red\n3 - green\n4 - yellow\n5 - gray\n6 - pink\n7 - orange\n8 - sky blue\n9 - maroon\n\nWhen solving an evaluation task, a test-taker has access to the training examples for the task (both the input and output grids), as well as the input grid of the test examples for the task. The test-taker must construct on its own the output grid corresponding to the input grid of each test example. “Constructing the output grid” is done entirely from scratch, meaning that the test-taker must decide what the height and width of the output grid should be, what colors they should place on the grid, and where. The task is successfully solved if the test-taker can produce the exact correct answer on all test examples for the task (binary measure of success).\n\nThe mapping from integers (0-9) to color names is arbitrary and should be treated as categorical labels; the transformation rule never relies on the specific integer–color pairing.\n\nARC AGI puzzle solutions feel "natural" to humans. Natural rules tend to build on visual priors and small-number counting. They are visually immediate or prominent for humans. That is why humans from all educational backgrounds are able to solve these puzzles. Rules that depend on non-visual considerations (e.g. mathematics that\'s more advanced than small number counting) or inelegant mechanics (e.g. tie-breakers, complex conditional rules, special exceptions) are considered "unnatural."  \n\nThis doesn\'t mean puzzles have to be simple or easy. Common patterns that puzzles use to increase difficulty are:  \n\n- **Rule composition:** multiple simple, natural rules are applied in sequence, with one transformation setting up the next.\n- **Contextual application:** a simple if/else based on a visually clear feature.\n- **In-grid symbol definition:** certain shapes or patterns act as symbols whose meaning is established using the training examples and which control some aspect of the transformation. The meaning can even vary across training examples.\n- **Iterated steps:** a simple action is applied repeatedly (as in assembling pieces of a jigsaw), where each step constrains the next. Repetition is restricted to a small number.\n- **Approximate features**: Features that are easily detected by humans, but hard to detect using naive algorithms. Examples are slightly broken symmetry, slightly broken continuity, off-center symmetry axes, objects defined via 8-connectivity etc.\n- **Non-locality:** the solution requires applying changes to one region, with the changes depending on features in a different, distant region.\n\n# Puzzle to solve\n\n{{puzzle}}\n\n# Task aids\n\nYou may execute code in the Jupyter notebook environment.\n\nVariables available in the Jupyter notebook:\n\n- `puzzle`: A Python `dict` containing the training and test grids. The key "train" contains the training examples in a list. Similarly, the key "test" contains the test examples as a list. Each training/test example is a dict containing keys "input" and "output". The values are the grids expressed as list of lists of integers (0 - 9). Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.\n\nYour very first code execution should be `puzzle.keys()` to check data access. \n\nDon\'t print the `puzzle` dict, because it prints out the grids that are already present in this user message and consumes valuable token budget.\n\n# Your task\n\nPlease solve the puzzle. Solving the puzzle means constructing a function `solution(grid: list[list[int]])` that implements the transformation from input grid to output grid shown in the training examples. The solution must work on all training examples.\n\n```\nfor idx, example in enumerate(puzzle["train"]):\n    assert solution(example["input"]) == example["output"], f"Train example {idx}: mismatch"\n```\n\nThe solution must generalize to the test input(s). Solutions that cheat by hardcoding the training example outputs are not acceptable.\n\n# Output format (STRICT)\n\nYour response must contain a single Python code block fenced by ```python ... ```. Must include a `solution` function. Must be self-contained i.e. include all necessary imports and helper functions.'
PYTHON_TOOL = {
    "type":"function",
    "function":{
        "name":"python",
        "description":"Execute Python in a persistent per-puzzle REPL and return stdout or a concise error.",
        "parameters":{"type":"object","properties":{"code":{"type":"string"}},"required":["code"],"additionalProperties":False},
    },
}
PIV_PROMPT_SOURCE_SHA256 = stable_json_sha256({"system":PIV_SOURCE_SYSTEM_PROMPT,"agentic":PIV_SOURCE_AGENTIC_PROMPT_TEMPLATE,"tool":PYTHON_TOOL})

def _common_prefix_len(a,b):
    n=min(len(a),len(b)); i=0
    while i<n and a[i]==b[i]: i+=1
    return i

if USE_MODEL:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=False)
    _template_path = MODEL_PATH / "chat_template.jinja"
    CHAT_TEMPLATE_SHA256 = small_file_sha256(_template_path) if _template_path.exists() else None
    with open(EVAL_CHALLENGES_PATH, "r", encoding="utf-8") as f:
        eval_challenges = json.load(f)

    _source_lib_sentence=(
        "The environment supports common third party libraries like numpy, scipy, shapely, networkx, "
        "scikit-image, more-itertools, pillow, python-constraint, ortools and z3-solver."
    )
    _local_lib_sentence=(
        "For this LOCAL throughput/PIV profile, assume the same approved P0 scientific Python environment used by the solver; "
        "do not rely on optional libraries not present in the mounted runtime."
    )
    SYSTEM_PROMPT=PIV_SOURCE_SYSTEM_PROMPT.replace(_source_lib_sentence,_local_lib_sentence).replace(
        "time out after 120.0 seconds", "time out after 10.0 seconds"
    )
    SYSTEM_PROMPT += "\n\n### Local P0 compute budget\nUse Python calls deliberately; this throughput probe preserves the solver's tool-enabled chat transport."

    def grid_lines(grid):
        return "\n".join(" ".join(str(int(x)) for x in row) for row in grid)
    def puzzle_to_source_text(task):
        parts=["## Training examples"]
        for i,ex in enumerate(task["train"]):
            parts += [f"### Training example {i}","Input:",grid_lines(ex["input"]),"Output:",grid_lines(ex["output"])]
        parts.append("## Tests" if len(task["test"])>1 else "## Test")
        for i,ex in enumerate(task["test"]):
            if len(task["test"])>1: parts.append(f"### Test {i}")
            parts += ["Input:",grid_lines(ex["input"])]
        return "\n\n".join(parts)
    def build_user_prompt(task):
        prompt=PIV_SOURCE_AGENTIC_PROMPT_TEMPLATE.replace("{{puzzle}}",puzzle_to_source_text(task))
        prompt=prompt.replace(
            'Each training/test example is a dict containing keys "input" and "output". The values are the grids expressed as list of lists of integers (0 - 9). Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.',
            'Each training example is a dict containing keys "input" and "output"; each test example contains only "input". The values are grids expressed as list of lists of integers (0 - 9). Access: `puzzle["train"][i]["input"]`, `puzzle["train"][i]["output"]`, `puzzle["test"][j]["input"]`.'
        )
        return prompt
    def render_chat(messages, add_generation_prompt=True):
        kwargs=dict(tokenize=False,add_generation_prompt=add_generation_prompt,tools=[PYTHON_TOOL],enable_thinking=True,reasoning_effort="xhigh",preserve_thinking=True)
        try:
            return tokenizer.apply_chat_template(messages,**kwargs),"qwen_tools+xhigh+preserve_thinking"
        except TypeError as exc:
            kwargs.pop("preserve_thinking",None)
            text=tokenizer.apply_chat_template(messages,**kwargs)
            return text,f"qwen_tools+xhigh(template-preserve-default);fallback={type(exc).__name__}"

    _assistant_tool_msg={
        "role":"assistant","content":"","reasoning_content":"I will inspect the puzzle with Python before finalizing the rule.",
        "tool_calls":[{"type":"function","function":{"name":"python","arguments":{"code":"print(puzzle.keys())"}}}],
    }
    _all_records=[]
    for task_id in sorted(eval_challenges):
        user_prompt=build_user_prompt(eval_challenges[task_id])
        base_messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user_prompt}]
        text,method=render_chat(base_messages,True)
        history_messages=base_messages+[_assistant_tool_msg]
        history_seed,_=render_chat(history_messages,False)
        variants={
            "small": PREFIX_EXTENSION_SMALL_TEXT.strip(),
            "medium": PREFIX_EXTENSION_TEXT.strip(),
            "large": PREFIX_EXTENSION_LARGE_TEXT.strip(),
            "agent": PREFIX_AGENT_TURN_TEXT.strip(),
        }
        rendered_ext={}
        for key,tool_text in variants.items():
            rendered_ext[key],_=render_chat(history_messages+[{"role":"tool","content":tool_text}],True)
        ids=tokenizer.encode(text,add_special_tokens=False)
        hist_ids=tokenizer.encode(history_seed,add_special_tokens=False)
        ext_ids={k:tokenizer.encode(v,add_special_tokens=False) for k,v in rendered_ext.items()}
        rec={"task_id":task_id,"prompt":text,"prompt_tokens":len(ids),"history_seed":history_seed,"history_seed_tokens":len(hist_ids),
             "prompt_extension_small":rendered_ext["small"],"prompt_extension_small_tokens":len(ext_ids["small"]),
             "prompt_extension":rendered_ext["medium"],"prompt_extension_tokens":len(ext_ids["medium"]),
             "prompt_extension_large":rendered_ext["large"],"prompt_extension_large_tokens":len(ext_ids["large"]),
             "prompt_extension_agent":rendered_ext["agent"],"prompt_extension_agent_tokens":len(ext_ids["agent"]),
             "shared_prefix_tokens_small":_common_prefix_len(hist_ids,ext_ids["small"]),
             "shared_prefix_tokens_medium":_common_prefix_len(hist_ids,ext_ids["medium"]),
             "shared_prefix_tokens_large":_common_prefix_len(hist_ids,ext_ids["large"]),
             "shared_prefix_tokens_agent":_common_prefix_len(hist_ids,ext_ids["agent"]),
             "extension_small_added_tokens":max(0,len(ext_ids["small"])-len(hist_ids)),
             "extension_added_tokens":max(0,len(ext_ids["medium"])-len(hist_ids)),
             "extension_large_added_tokens":max(0,len(ext_ids["large"])-len(hist_ids)),
             "extension_agent_added_tokens":max(0,len(ext_ids["agent"])-len(hist_ids)),
             "render_method":method}
        _all_records.append(rec)
else:
    CHAT_TEMPLATE_SHA256="SELFTEST_SYNTHETIC"
    _all_records=[]
    for i in range(max(160,PROMPT_POOL_SIZE)):
        n_tok=512+(i*137)%9500
        text=(f"<|synthetic_arc_prompt_{i}|> "+("x "*n_tok)).strip()
        history_seed=text+" <tool_call>python</tool_call>"
        htok=n_tok+16
        adds={"small":54,"medium":1335,"large":5270,"agent":900}
        rec={"task_id":f"synthetic_{i:03d}","prompt":text,"prompt_tokens":n_tok,"history_seed":history_seed,"history_seed_tokens":htok,"render_method":"synthetic"}
        for key,field in [("small","prompt_extension_small"),("medium","prompt_extension"),("large","prompt_extension_large"),("agent","prompt_extension_agent")]:
            add=adds[key]; rec[field]=history_seed+(" y"*add); rec[field+"_tokens"]=htok+add
            rec["shared_prefix_tokens_"+("medium" if key=="medium" else key)]=htok
        rec.update({"extension_small_added_tokens":adds["small"],"extension_added_tokens":adds["medium"],"extension_large_added_tokens":adds["large"],"extension_agent_added_tokens":adds["agent"]})
        _all_records.append(rec)

_max_future=lambda r:max(r["prompt_tokens"],r["history_seed_tokens"],r["prompt_extension_small_tokens"],r["prompt_extension_tokens"],r["prompt_extension_large_tokens"],r["prompt_extension_agent_tokens"])
_fit=[r for r in _all_records if _max_future(r)+max(OUTPUT_TOKENS,ANCHOR_OUTPUT_TOKENS,SOLVER_LONG_OUTPUT_TOKENS,AGENT_TURN_OUTPUT_TOKENS)+64<=MAX_MODEL_LEN]
_fit.sort(key=lambda r:(r["prompt_tokens"],r["task_id"]))
if len(_fit)<=PROMPT_POOL_SIZE: PROMPT_POOL=list(_fit)
else:
    _idx=np.linspace(0,len(_fit)-1,PROMPT_POOL_SIZE).round().astype(int)
    PROMPT_POOL=[_fit[i] for i in sorted(set(int(i) for i in _idx))]
PROMPT_POOL.sort(key=lambda r:(r["prompt_tokens"],r["task_id"]))
PROMPTS_PATH=WORK_ROOT/"prompts.json"
write_json(PROMPTS_PATH,PROMPT_POOL)
_lens=np.array([r["prompt_tokens"] for r in PROMPT_POOL])
PROMPT_POOL_STATS={
    "prompt_profile":"piv_source_aligned_qwen_tools_xhigh" if USE_MODEL else "synthetic",
    "piv_prompt_source_sha256":PIV_PROMPT_SOURCE_SHA256,"eval_tasks_total":len(_all_records),"dropped_for_context":len(_all_records)-len(_fit),"pool_size":len(PROMPT_POOL),
    "prompt_tokens_min":int(_lens.min()),"prompt_tokens_median":float(np.median(_lens)),"prompt_tokens_p90":float(np.percentile(_lens,90)),"prompt_tokens_max":int(_lens.max()),
    "render_methods":sorted(set(r["render_method"] for r in PROMPT_POOL)),
    "history_seed_tokens_median":float(np.median([r["history_seed_tokens"] for r in PROMPT_POOL])),
    "shared_prefix_tokens_agent_median":float(np.median([r["shared_prefix_tokens_agent"] for r in PROMPT_POOL])),
    "extension_added_tokens_small_median":float(np.median([r["extension_small_added_tokens"] for r in PROMPT_POOL])),
    "extension_added_tokens_medium_median":float(np.median([r["extension_added_tokens"] for r in PROMPT_POOL])),
    "extension_added_tokens_large_median":float(np.median([r["extension_large_added_tokens"] for r in PROMPT_POOL])),
    "extension_added_tokens_agent_median":float(np.median([r["extension_agent_added_tokens"] for r in PROMPT_POOL])),
    "chat_template_sha256":CHAT_TEMPLATE_SHA256,"prompts_sha256":small_file_sha256(PROMPTS_PATH),
}
if USE_MODEL and not all(str(x).startswith("qwen_tools+xhigh") for x in PROMPT_POOL_STATS["render_methods"]):
    raise RuntimeError(f"Real GPU prompt render must be Qwen tool/xhigh template only: {PROMPT_POOL_STATS['render_methods']}")
if len(PROMPT_POOL)<16: raise RuntimeError("Prompt pool too small for balanced throughput probe.")

def balanced_indices(pool_len,n,salt=0):
    if n<=0:return []
    base=[int(round(x)) for x in np.linspace(0,pool_len-1,min(n,pool_len))]
    if n>pool_len: base += [base[(j+salt)%len(base)] for j in range(n-pool_len)]
    if base and salt:
        k=salt%len(base); base=base[k:]+base[:k]
    return base
_all_ns=sorted({ANCHOR_BATCH_N,MIXED_BATCH_N,8,16,32,64,128}|{int(n) for c in CONFIG_LADDER for n in c.get("decode_points",[])})
BATCH_PLAN={str(n):balanced_indices(len(PROMPT_POOL),n,salt=n//4) for n in _all_ns}
write_json(WORK_ROOT/"measurement"/"batch_plan.json",BATCH_PLAN)
_batch_diag=[]
for n,idxs in BATCH_PLAN.items():
    ls=[PROMPT_POOL[i]["prompt_tokens"] for i in idxs]
    _batch_diag.append({"n":int(n),"count":len(idxs),"min":min(ls),"median":float(np.median(ls)),"p90":float(np.percentile(ls,90)),"max":max(ls),"unique_indices":len(set(idxs))})
write_json(WORK_ROOT/"measurement"/"batch_plan_diagnostics.json",_batch_diag)
# One globally balanced 64-prompt workload, split exactly once into disjoint TP2 replica halves.
_pair_common=balanced_indices(len(PROMPT_POOL),TP2_PAIR_N*2,salt=117)
if len(_pair_common)!=TP2_PAIR_N*2 or len(set(_pair_common))!=len(_pair_common):
    raise RuntimeError(f"TP2x2 pair workload must contain {TP2_PAIR_N*2} unique prompt indices; got {len(_pair_common)} / {len(set(_pair_common))}")
TP2_PAIR_PROMPT_PLAN={
    "common_indices":_pair_common,
    "common_task_ids":[PROMPT_POOL[i]["task_id"] for i in _pair_common],
    "replica_indices":{"0":_pair_common[:TP2_PAIR_N],"1":_pair_common[TP2_PAIR_N:]},
    "replica_task_ids":{"0":[PROMPT_POOL[i]["task_id"] for i in _pair_common[:TP2_PAIR_N]],"1":[PROMPT_POOL[i]["task_id"] for i in _pair_common[TP2_PAIR_N:]]},
}
if set(TP2_PAIR_PROMPT_PLAN["replica_indices"]["0"]) & set(TP2_PAIR_PROMPT_PLAN["replica_indices"]["1"]):
    raise RuntimeError("TP2x2 replica prompt sets overlap")
write_json(WORK_ROOT/"measurement"/"tp2_pair_prompt_plan.json",TP2_PAIR_PROMPT_PLAN)
print(json.dumps(PROMPT_POOL_STATS,indent=2)); print("Balanced batch diagnostics:",_batch_diag); print("TP2x2 disjoint prompt plan:", {k:len(v) if isinstance(v,list) else None for k,v in TP2_PAIR_PROMPT_PLAN.items()})


{
  "prompt_profile": "piv_source_aligned_qwen_tools_xhigh",
  "piv_prompt_source_sha256": "6416f971f993b5ba5b72e7654d856967020825bd4fb8f96079cfe3c6dede837f",
  "eval_tasks_total": 120,
  "dropped_for_context": 0,
  "pool_size": 96,
  "prompt_tokens_min": 2835,
  "prompt_tokens_median": 6457.0,
  "prompt_tokens_p90": 10375.5,
  "prompt_tokens_max": 17803,
  "render_methods": [
    "qwen_tools+xhigh+preserve_thinking"
  ],
  "history_seed_tokens_median": 6504.0,
  "shared_prefix_tokens_agent_median": 6504.0,
  "extension_added_tokens_small_median": 31.0,
  "extension_added_tokens_medium_median": 1109.0,
  "extension_added_tokens_large_median": 4373.0,
  "extension_added_tokens_agent_median": 702.0,
  "chat_template_sha256": "c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041",
  "prompts_sha256": "9bf6236c6c7ce6929375b5a915c836c341a1e7aa2e21f8ecef35d8ea3d0fc350"
}
Balanced batch diagnostics: [{'n': 8, 'count': 8, 'min': 2835, 'median': 6330.5, 'p90': 11897.8, 'max': 17803,

In [4]:
# =====================================================================================
# CELL 4 — WORKER SCRIPT (one engine per subprocess) + PARENT ORCHESTRATION HELPERS
# The parent (this notebook) never imports torch/vllm, so it holds no CUDA context and every
# configuration starts from empty GPUs. Timeouts kill the whole process group, then nvidia-smi
# is polled until the GPUs are free again before the next configuration starts.
# =====================================================================================
WORKER_SRC = r'''
# tp_worker.py -- measure one vLLM engine configuration. Written by the TP-Q27 v1.2.4 MASTER probe notebook.
import argparse, json, os, sys, time, random, traceback, dataclasses

class Unsupported(Exception):
    pass

def read_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

class FakeEngine:
    """CPU stand-in with a plausible speed model; only for SELFTEST_CPU."""
    def __init__(self, spec, result):
        kw = dict(spec["engine_kwargs"])
        name = spec["name"]
        if "fake_fail" in name:
            raise RuntimeError("selftest: simulated engine init failure (CUDA OOM during graph capture)")
        if "fake_hang" in name:
            time.sleep(3600)
        self.scale = float(spec.get("fake_time_scale", 1.0))
        base = 6.25
        if not kw.get("enforce_eager", True):
            base *= 2.6
        if not kw.get("disable_custom_all_reduce", True):
            base *= 1.08
        if kw.get("kv_cache_dtype") == "fp8":
            base *= 0.97
        self.base = base
        self.prefix = bool(kw.get("enable_prefix_caching"))
        self.seen = set()
        self.seen_texts = set()
        time.sleep(0.3 * 1.0)
        result["engine_kwargs_used"] = kw
        result["load_sec"] = 277.0 + (90.0 if not kw.get("enforce_eager", True) else 0.0)
        result["fake"] = True

    def generate(self, prompts, sp):
        n=len(prompts); splist=sp if isinstance(sp,list) else [sp]*n
        mts=[int(x["max_tokens"]) for x in splist]; per_seq=self.base/(1.0+0.012*n)
        prior=list(self.seen_texts)
        def cp_chars(a,b):
            k=0; m=min(len(a),len(b))
            while k<m and a[k]==b[k]: k+=1
            return k
        cached=[]; prefill_each=[]
        for p in prompts:
            c=max([cp_chars(p,q) for q in prior] or [0]) if self.prefix else 0
            c_tok=min(len(p)//4,c//4); cached.append(c_tok)
            uncached=max(1,len(p)//4-c_tok); rate=40000.0 if c_tok>0 else 4000.0
            prefill_each.append(uncached/rate)
        self.seen_texts.update(prompts); self.seen.update(hash(p) for p in prompts)
        prefill=sum(prefill_each); decode_work=sum(mts)/max(1.0,n*per_seq)
        time.sleep((prefill+decode_work)/self.scale)
        outs=[]
        for j,(p,one) in enumerate(zip(prompts,splist)):
            mt=int(one["max_tokens"]); rng=random.Random(int(one.get("seed",0))+n+j)
            ids=[rng.randrange(1000,50000) for _ in range(mt)] if float(one.get("temperature",1.0))>0 else [(i*7919+len(p))%50000 for i in range(mt)]
            dec=max(0.0,(mt-1)/max(1e-9,per_seq)); ttft=prefill_each[j]
            outs.append({"request_id":f"fake-{j}","finished":True,"finish_reason":"length","stop_reason":None,
                         "prompt_token_ids":[0]*(len(p)//4),"token_ids":ids,"num_cached_tokens":cached[j],
                         "num_cache_creation_tokens":max(0,len(p)//4-cached[j]),
                         "metrics":{"first_token_latency":ttft,"queued_ts":0.0,"scheduled_ts":0.0,"first_token_ts":ttft,"last_token_ts":ttft+dec,"num_generation_tokens":mt,"is_corrupted":False}})
        return outs

    def reset_prefix_cache(self):
        self.seen_texts.clear(); self.seen.clear(); return True


class RealEngine:
    def __init__(self, spec, result):
        import inspect
        import vllm
        from vllm import LLM, SamplingParams
        from vllm.engine.arg_utils import EngineArgs
        self.SamplingParams = SamplingParams
        result["runtime"] = {"vllm": getattr(vllm, "__version__", None)}
        valid = {f.name for f in dataclasses.fields(EngineArgs)} | set(inspect.signature(LLM.__init__).parameters)
        valid.discard("kwargs"); valid.discard("self")
        kw = dict(spec["engine_kwargs"])
        overrides = dict(spec.get("overrides", {}))
        if spec.get("require_request_metrics") and kw.get("disable_log_stats") is not False:
            raise RuntimeError("instrumented row requires disable_log_stats=False")
        # The Q1-frozen base kwargs are known-good on this stack; only the ladder's override keys can be
        # unsupported. A rejected override is a distinct outcome ("unsupported"), not an engine failure.
        unsupported = sorted(k for k in overrides if k not in valid)
        result["unsupported_kwargs"] = unsupported
        result["base_kwargs_not_in_signature"] = sorted(k for k in kw if k not in valid and k not in overrides)
        if unsupported:
            raise Unsupported("engine kwargs not accepted by this vLLM build: %s" % unsupported)
        result["engine_kwargs_used"] = kw
        t0 = time.perf_counter()
        try:
            self.llm = LLM(**kw)
        except Exception as e:
            msg = "%s: %s" % (type(e).__name__, e)
            if overrides and any(k in msg for k in overrides):
                result["unsupported_kwargs"] = sorted(k for k in overrides if k in msg)
                raise Unsupported(msg)
            raise
        result["load_sec"] = time.perf_counter() - t0
        result["request_metrics_enabled"] = (kw.get("disable_log_stats") is False)

    def reset_prefix_cache(self):
        if not hasattr(self.llm, "reset_prefix_cache"):
            raise Unsupported("vLLM LLM.reset_prefix_cache() unavailable on this runtime")
        return bool(self.llm.reset_prefix_cache())

    def generate(self, prompts, sp):
        if isinstance(sp,list): params=[self.SamplingParams(**x) for x in sp]
        else: params=self.SamplingParams(**sp)
        outs=self.llm.generate(prompts,params,use_tqdm=False)
        rows=[]
        for o in outs:
            m=getattr(o,"metrics",None)
            md=None
            if m is not None:
                md={k:getattr(m,k,None) for k in ("arrival_time","queued_ts","scheduled_ts","first_token_ts","last_token_ts","first_token_latency","num_generation_tokens","is_corrupted")}
            oo=o.outputs[0]
            rows.append({"request_id":getattr(o,"request_id",None),"finished":getattr(o,"finished",None),
                         "finish_reason":getattr(oo,"finish_reason",None),"stop_reason":getattr(oo,"stop_reason",None),
                         "prompt_token_ids":list(getattr(o,"prompt_token_ids",None) or []),
                         "token_ids":list(oo.token_ids),
                         "num_cached_tokens":getattr(o,"num_cached_tokens",None),
                         "num_cache_creation_tokens":getattr(o,"num_cache_creation_tokens",None),
                         "metrics":md})
        return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--spec", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--fake", action="store_true")
    a = ap.parse_args()
    spec = read_json(a.spec)
    scale = float(spec.get("fake_time_scale", 1.0)) if a.fake else 1.0
    result = {"config": spec["name"], "status": "started", "phases": [], "timeline": [], "error": None,
              "engine_kwargs_used": None, "unsupported_kwargs": [], "load_sec": None, "fingerprint": None,
              "worker_pid": os.getpid(), "started_epoch": time.time()}

    def dump():
        tmp = a.out + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, sort_keys=True, default=str)
        os.replace(tmp, a.out)

    def mark(event):
        result["timeline"].append({"event": event, "epoch": time.time()})
        dump()

    dump()
    pool = read_json(spec["prompts_path"])
    texts = [r["prompt"] for r in pool]
    ptoks = [int(r["prompt_tokens"]) for r in pool]
    history_texts=[r.get("history_seed",r["prompt"]) for r in pool]
    history_ptoks=[int(r.get("history_seed_tokens",r["prompt_tokens"])) for r in pool]
    ext_small_texts=[r.get("prompt_extension_small",r["prompt"]) for r in pool]
    ext_small_ptoks=[int(r.get("prompt_extension_small_tokens",r["prompt_tokens"])) for r in pool]
    ext_texts=[r.get("prompt_extension",r["prompt"]) for r in pool]
    ext_ptoks=[int(r.get("prompt_extension_tokens",r["prompt_tokens"])) for r in pool]
    ext_large_texts=[r.get("prompt_extension_large",r["prompt"]) for r in pool]
    ext_large_ptoks=[int(r.get("prompt_extension_large_tokens",r["prompt_tokens"])) for r in pool]
    ext_agent_texts=[r.get("prompt_extension_agent",r["prompt"]) for r in pool]
    ext_agent_ptoks=[int(r.get("prompt_extension_agent_tokens",r["prompt_tokens"])) for r in pool]
    shared_small=[int(r.get("shared_prefix_tokens_small",0)) for r in pool]
    shared_medium=[int(r.get("shared_prefix_tokens_medium",0)) for r in pool]
    shared_large=[int(r.get("shared_prefix_tokens_large",0)) for r in pool]
    shared_agent=[int(r.get("shared_prefix_tokens_agent",0)) for r in pool]
    out_tokens = int(spec["output_tokens"])
    seed = int(spec["seed"])
    engine = None
    result["cache_resets"] = []

    def reset_cache(tag):
        ok=engine.reset_prefix_cache()
        result["cache_resets"].append({"tag":tag,"ok":bool(ok),"epoch":time.time()}); dump()
        if not ok: raise RuntimeError(f"prefix cache reset failed at {tag}")
        return True

    def _finalize_early_success():
        # Early-success branches (drift sentinel / TP2x2 member) must get the same cleanup
        # protection as the normal path. Start the watchdog BEFORE dropping the engine so a
        # stuck vLLM destructor / atexit path cannot hold the parent until the full config timeout.
        nonlocal engine
        import threading, gc
        threading.Thread(target=lambda: (time.sleep(90.0), os._exit(0)), daemon=True).start()
        engine = None
        gc.collect()
        sys.stdout.flush(); sys.stderr.flush()

    def wave(name,idx,sp,note=None,text_source=None,ptok_source=None,base_ptok_source=None,shared_prefix_source=None):
        source=texts if text_source is None else text_source; psource=ptoks if ptok_source is None else ptok_source
        bsource=None if base_ptok_source is None else base_ptok_source; shsource=None if shared_prefix_source is None else shared_prefix_source
        ts=[source[i%len(source)] for i in idx]; prompt_lengths=[int(psource[i%len(psource)]) for i in idx]
        base_lengths=[int(bsource[i%len(bsource)]) for i in idx] if bsource is not None else None
        shared_lengths=[int(shsource[i%len(shsource)]) for i in idx] if shsource is not None else None
        pt_expected=sum(prompt_lengths); requested=([int(x["max_tokens"]) for x in sp] if isinstance(sp,list) else [int(sp["max_tokens"])]*len(ts))
        mark("phase_start:"+name); t0=time.perf_counter(); outs=engine.generate(ts,sp); wall=(time.perf_counter()-t0)*scale; mark("phase_end:"+name)
        if len(outs)!=len(ts): raise RuntimeError(f"{name}: output-count mismatch {len(outs)} != {len(ts)}")
        ot=[len(o["token_ids"]) for o in outs]; pe=[len(o["prompt_token_ids"]) for o in outs]
        cached=[o.get("num_cached_tokens") for o in outs]; cached_valid=[int(x) for x in cached if x is not None]
        metrics=[o.get("metrics") or {} for o in outs]
        finished=[o.get("finished") for o in outs]; finish_reasons=[o.get("finish_reason") for o in outs]
        corrupted=[bool(m.get("is_corrupted")) for m in metrics]
        phase_integrity=(all(t==r for t,r in zip(ot,requested)) and all(x is not False for x in finished) and
                         all(x in (None,"length") for x in finish_reasons) and not any(corrupted))
        if not phase_integrity:
            raise RuntimeError(f"{name}: phase output integrity failed lengths={ot} requested={requested} finished={finished} reasons={finish_reasons} corrupted={corrupted}")
        def pct(a,q):
            a=sorted(float(x) for x in a if x is not None)
            if not a:return None
            pos=(len(a)-1)*q; lo=int(pos); hi=min(len(a)-1,lo+1); f=pos-lo
            return a[lo]*(1-f)+a[hi]*f
        ttft=[m.get("first_token_latency") for m in metrics if m.get("first_token_latency") is not None]
        qtime=[]; dtime=[]; req_tps=[]
        for m,tok in zip(metrics,ot):
            q0,q1=m.get("queued_ts"),m.get("scheduled_ts"); f,l=m.get("first_token_ts"),m.get("last_token_ts")
            if q0 is not None and q1 is not None and q1>=q0: qtime.append(q1-q0)
            if f is not None and l is not None and l>=f:
                d=l-f; dtime.append(d)
                if d>0 and tok>1: req_tps.append((tok-1)/d)
        pls=sorted(prompt_lengths)
        ph={"phase":name,"n":len(ts),"wall_sec":wall,"requested_output_tokens":requested,"requested_output_tokens_total":sum(requested),"max_tokens":max(requested) if requested else 0,
            "out_tokens_total":sum(ot),"out_tokens_list":ot,"out_tokens_min":min(ot),"out_tokens_max":max(ot),"all_reached_max_tokens":all(t==r for t,r in zip(ot,requested)),
            "phase_integrity":phase_integrity,"finished_list":finished,"finish_reasons":finish_reasons,"corrupted_count":sum(corrupted),
            "prompt_indices":[int(i%len(source)) for i in idx],"prompt_task_ids":[pool[int(i%len(source))]["task_id"] for i in idx],
            "prompt_tokens_total_tokenizer":pt_expected,"prompt_tokens_total_engine":(sum(pe) if all(pe) else None),
            "prompt_tokens_min":min(pls),"prompt_tokens_median":pct(pls,0.5),"prompt_tokens_p90":pct(pls,0.9),"prompt_tokens_max":max(pls),
            "prompt_unique_indices":len(set(i%len(source) for i in idx)),"prompts_repeated":(len(set(i%len(source) for i in idx))<len(idx)),"prompt_index_range":[min(idx),max(idx)] if idx else None,
            "agg_gen_tps":sum(ot)/wall if wall>0 else None,"per_seq_gen_tps":(sum(ot)/len(ot))/wall if wall>0 else None,
            "sec_per_output_token_per_seq":wall/max(1,max(requested) if requested else 1),"effective_full_prefill_tps":pt_expected/wall if wall>0 else None,"prefill_tps":pt_expected/wall if wall>0 else None,
            "num_cached_tokens_list":cached,"num_cached_tokens_total":sum(cached_valid) if cached_valid else None,
            "cache_hit_ratio_prompt":(sum(cached_valid)/pt_expected) if cached_valid and pt_expected else None,
            "ttft_sec_median":pct(ttft,0.5),"ttft_sec_p95":pct(ttft,0.95),"queue_sec_median":pct(qtime,0.5),"queue_sec_p95":pct(qtime,0.95),
            "request_decode_sec_median":pct(dtime,0.5),"request_decode_sec_p95":pct(dtime,0.95),"request_decode_tps_median":pct(req_tps,0.5),"request_decode_tps_p05":pct(req_tps,0.05),
            "request_timing_available_count":sum(1 for m in metrics if m and m.get("first_token_latency") is not None),"request_metrics":metrics,"note":note}
        if spec.get("require_request_metrics") and ph["request_timing_available_count"] != len(ts):
            raise RuntimeError(f"{name}: request timing unavailable for {len(ts)-ph['request_timing_available_count']} / {len(ts)} requests; disable_log_stats=False instrumentation failed")
        if base_lengths is not None:
            base_total=sum(base_lengths); new_total=max(0,pt_expected-base_total); ph["base_prompt_tokens_total"]=base_total; ph["new_suffix_tokens_total"]=new_total; ph["new_suffix_tps"]=new_total/wall if wall>0 else None
        if shared_lengths is not None:
            sh=sum(shared_lengths); ph["expected_shared_prefix_tokens_total"]=sh
            ph["cached_vs_expected_shared_ratio"]=(sum(cached_valid)/sh) if cached_valid and sh>0 else None
        result["phases"].append(ph); dump(); return outs


    def balanced(n, salt=0):
        # prompts.json is sorted by length. Quantile-spaced indices make n=8/16/32/... comparable.
        if n <= 0: return []
        m=len(texts); k=min(n,m)
        if k == 1: base=[0]
        else: base=[int(round(j*(m-1)/(k-1))) for j in range(k)]
        if n > m: base += [base[(j+salt)%len(base)] for j in range(n-m)]
        if base and salt:
            q=salt%len(base); base=base[q:]+base[:q]
        return base

    def balanced_from(candidates, n, salt=0):
        candidates=list(candidates)
        if not candidates or n<=0: return []
        k=min(n,len(candidates))
        if k==1: out=[candidates[0]]
        else: out=[candidates[int(round(j*(len(candidates)-1)/(k-1)))] for j in range(k)]
        if n>len(candidates): out += [out[(j+salt)%len(out)] for j in range(n-len(candidates))]
        if out and salt:
            q=salt%len(out); out=out[q:]+out[:q]
        return out

    try:
        mark("load_start")
        engine = FakeEngine(spec, result) if a.fake else RealEngine(spec, result)
        mark("load_end")
        dump()
        max_seqs = int(spec["engine_kwargs"].get("max_num_seqs", 16))
        base_sp = {"temperature": 1.0, "top_p": 0.95, "top_k": 20, "seed": seed, "ignore_eos": True}
        pool_len = len(texts)
        # warmup (also validates generate() on this engine)
        n_warm = min(int(spec["warmup_prompts"]), pool_len)
        _warm_texts=[t + "\n\n# throughput-warmup-only" for t in texts]
        _warm_ptoks=[x + 8 for x in ptoks]
        wave("warmup", balanced(n_warm, salt=5), dict(base_sp, max_tokens=16), text_source=_warm_texts, ptok_source=_warm_ptoks, note="cache-isolated warmup prompt variant")

        if spec.get("drift_sentinel"):
            wave("decode@8", balanced(8,salt=2), dict(base_sp,max_tokens=out_tokens), note="end-session frozen-Q1 drift sentinel")
            fk=min(int(spec["fingerprint_prompts"]),len(texts)); fp_sp={"temperature":0.0,"max_tokens":int(spec["fingerprint_tokens"]),"ignore_eos":True,"seed":seed}
            outs=wave("fingerprint",balanced(fk,salt=3),fp_sp); result["fingerprint"]=[o["token_ids"] for o in outs]; result["status"]="ok"
            result["finished_epoch"]=time.time(); dump(); print("TP_WORKER_DONE status=ok config=%s"%spec["name"],flush=True)
            _finalize_early_success(); return

        # Synchronized TP2x2 member: load/warm first, then wait for parent barrier and run one balanced decode wave.
        if spec.get("pair_member"):
            from pathlib import Path as _P
            bdir = _P(spec["barrier_dir"]); bdir.mkdir(parents=True, exist_ok=True)
            (_P(bdir) / ("ready_" + str(spec["pair_member_id"]))).write_text("ready")
            go = _P(bdir) / "GO"
            wait0 = time.time()
            while not go.exists():
                if time.time() - wait0 > float(spec.get("barrier_timeout_sec", 300)):
                    raise RuntimeError("pair barrier timeout")
                time.sleep(0.05 if a.fake else 0.2)
            n = min(int(spec["pair_n"]), max_seqs)
            idx=[int(i) for i in spec.get("pair_prompt_indices",[])]
            if len(idx)!=n or len(set(idx))!=n: raise RuntimeError(f"pair prompt plan invalid for member {spec.get('pair_member_id')}: {len(idx)} / {len(set(idx))}")
            wave("pair_decode@%d" % n, idx, dict(base_sp, max_tokens=out_tokens), note="synchronized TP2x2 member; globally disjoint prompt half")
            result["status"] = "ok"; result["finished_epoch"] = time.time(); dump()
            print("TP_WORKER_DONE status=ok config=%s" % spec["name"], flush=True)
            _finalize_early_success(); return

        if spec.get("anchor_long"):
            n_anchor = min(int(spec.get("anchor_batch_n", 8)), max_seqs)
            wave("anchor_long@%d" % n_anchor, balanced(n_anchor, salt=7),
                 dict(base_sp, max_tokens=int(spec.get("anchor_output_tokens", out_tokens))), note="Q1/PIV long-decode anchor")

        used_decode=set()
        for n in sorted(int(x) for x in spec["decode_points"]):
            if n > max_seqs:
                result["phases"].append({"phase": "decode@%d" % n, "n": n, "skipped": "n > max_num_seqs=%d" % max_seqs}); dump(); continue
            idx = balanced(n, salt=n//4)
            used_decode.update(i % pool_len for i in idx)
            wave("decode@%d" % n, idx, dict(base_sp, max_tokens=out_tokens), note="balanced prompt-length quantiles")

        # Solver-representative long decode: PIV's active reasoning budget is 3,072 model tokens/turn.
        # Only selected configs carry solver_long_n so this does not multiply every optional row's cost.
        if spec.get("solver_long_n"):
            ln=min(int(spec["solver_long_n"]),max_seqs)
            wave("solver_long@%d" % ln, balanced(ln,salt=61),
                 dict(base_sp,max_tokens=int(spec.get("solver_long_tokens",3072))),
                 note="solver-representative 3072-token decode")

        # Mixed completion lengths: controlled continuous-batching tail, independent of EOS behavior.
        mn=min(int(spec.get("mixed_batch_n", 16)), max_seqs)
        if bool(spec["engine_kwargs"].get("enable_prefix_caching")):
            _fresh_mixed=[i for i in range(pool_len) if i not in used_decode]
            midx=balanced_from(_fresh_mixed if len(_fresh_mixed)>=mn else range(pool_len), mn, salt=53)
            _mixed_note="controlled heterogeneous max_tokens; fresh prompts for prefix-cache row" if len(_fresh_mixed)>=mn else "controlled heterogeneous max_tokens; prefix-cache overlap unavoidable"
        else:
            midx=balanced(mn, salt=53); _mixed_note="controlled heterogeneous max_tokens; continuous-batching tail"
        fracs=list(spec.get("mixed_output_fractions", [0.25,0.5,0.75,1.0]))
        mixed_sp=[]
        for j in range(mn):
            cap=max(1,int(round(out_tokens*float(fracs[j%len(fracs)]))))
            mixed_sp.append(dict(base_sp, max_tokens=cap, seed=seed+j+10000))
        wave("mixed_decode@%d" % mn, midx, mixed_sp, note=_mixed_note)

        # Repeatability / jitter at one balanced point. Prefix-cache rows are excluded to avoid measuring warmed cache.
        if not bool(spec["engine_kwargs"].get("enable_prefix_caching")):
            sn=min(int(spec.get("stability_batch_n", 32)), max_seqs)
            for rr in range(int(spec.get("stability_repeats", 2))):
                wave("stability@%d_r%d" % (sn,rr+1), balanced(sn, salt=71+rr*11),
                     dict(base_sp, max_tokens=int(spec.get("stability_tokens", 256)), seed=seed+20000+rr), note="repeatability/jitter wave")

        # Base cold/exact-repeat plus matched cache A/B and realistic next-turn extensions.
        k=min(int(spec["prefill_prompts"]),pool_len,max_seqs)
        pre_idx=balanced(k,salt=91)
        pre_sp={"temperature":0.0,"max_tokens":1,"ignore_eos":True}
        prefix_on=bool(spec["engine_kwargs"].get("enable_prefix_caching"))
        # A true cold cache phase requires an explicit cache reset when prefix caching is enabled.
        if prefix_on: reset_cache("before_prefill_cold")
        wave("prefill_cold",pre_idx,pre_sp,note="explicit cache reset before cold phase" if prefix_on else "prefix caching disabled; no reusable prefix cache")
        wave("prefill_warm_exact",pre_idx,pre_sp,note="identical prompts re-sent",shared_prefix_source=ptoks)
        if prefix_on:
            # Matched A/B: exact same medium extension prompt uncached vs history-seeded cached, >=3 repeats.
            for rr in range(int(spec.get("cache_ab_repeats",3))):
                reset_cache(f"cache_ab_uncached_r{rr+1}")
                wave(f"cache_ab_uncached_medium_r{rr+1}",pre_idx,pre_sp,note="matched cache A/B uncached medium extension",
                     text_source=ext_texts,ptok_source=ext_ptoks,base_ptok_source=history_ptoks,shared_prefix_source=shared_medium)
                reset_cache(f"cache_ab_cached_r{rr+1}")
                wave(f"tool_history_seed_ab_r{rr+1}",pre_idx,pre_sp,note="matched cache A/B history seed",text_source=history_texts,ptok_source=history_ptoks)
                wave(f"cache_ab_cached_medium_r{rr+1}",pre_idx,pre_sp,note="matched cache A/B cached medium extension",
                     text_source=ext_texts,ptok_source=ext_ptoks,base_ptok_source=history_ptoks,shared_prefix_source=shared_medium)
            # Small and large extension cache-hit evidence each start from a clean cache + identical history seed.
            for label,src,pts,shared in [("small",ext_small_texts,ext_small_ptoks,shared_small),("large",ext_large_texts,ext_large_ptoks,shared_large)]:
                reset_cache(f"before_extension_{label}")
                wave(f"tool_history_seed_{label}",pre_idx,pre_sp,note=f"history seed for {label} cached extension",text_source=history_texts,ptok_source=history_ptoks)
                wave(f"prefill_warm_extension_{label}",pre_idx,pre_sp,note=f"cached tool-history + {label} tool result",text_source=src,ptok_source=pts,base_ptok_source=history_ptoks,shared_prefix_source=shared)
            if spec.get("agent_turn_n"):
                reset_cache("before_agent_turn")
                wave("tool_history_seed_agent",pre_idx,pre_sp,note="history seed for cached next-agent-turn decode",text_source=history_texts,ptok_source=history_ptoks)
                an=min(int(spec["agent_turn_n"]),len(pre_idx),max_seqs); aidx=pre_idx[:an]
                wave("agent_turn_extension@%d"%an,aidx,dict(base_sp,max_tokens=int(spec.get("agent_turn_output_tokens",256)),seed=seed+31000),
                     note="cached tool-history + new tool result + next assistant decode",text_source=ext_agent_texts,ptok_source=ext_agent_ptoks,base_ptok_source=history_ptoks,shared_prefix_source=shared_agent)
        else:
            result["phases"].append({"phase":"prefill_extensions","skipped":"enable_prefix_caching=False: matched cache A/B and extension waves are not meaningful"}); dump()

        fk=min(int(spec["fingerprint_prompts"]),len(texts))
        fp_sp={"temperature":0.0,"max_tokens":int(spec["fingerprint_tokens"]),"ignore_eos":True,"seed":seed}
        outs=wave("fingerprint", balanced(fk,salt=3), fp_sp)
        result["fingerprint"]=[o["token_ids"] for o in outs]
        result["status"] = "ok"
    except Unsupported as e:
        result["status"] = "unsupported"
        result["error"] = str(e)
    except BaseException as e:
        result["status"] = "failed"
        result["error"] = "%s: %s" % (type(e).__name__, e)
        result["traceback"] = traceback.format_exc()
    result["finished_epoch"] = time.time()
    dump()
    code = 0 if result["status"] in ("ok", "unsupported") else 3
    print("TP_WORKER_DONE status=%s config=%s" % (result["status"], spec["name"]), flush=True)
    # Prefer a clean exit (vLLM's atexit hooks shut the TP workers down); if that hangs, hard-exit
    # after 90 s. The parent reclaims any GPU process that survives either path.
    import threading
    threading.Thread(target=lambda: (time.sleep(90.0), os._exit(code)), daemon=True).start()
    engine = None
    import gc
    gc.collect()
    sys.stdout.flush(); sys.stderr.flush()
    sys.exit(code)

if __name__ == "__main__":
    main()
'''
WORKER_PATH = WORK_ROOT / "tp_worker.py"
WORKER_PATH.write_text(WORKER_SRC, encoding="utf-8")
WORKER_SHA256 = small_file_sha256(WORKER_PATH)
import py_compile
py_compile.compile(str(WORKER_PATH), doraise=True)
print("worker script:", WORKER_PATH, "sha256", WORKER_SHA256[:16], "compile OK")

# ---------------------------------------------------------------- parent helpers
def gpu_sample():
    fields=list(GPU_TELEMETRY_FIELDS)
    rows=nvidia_smi_query(fields)
    if rows is None:
        fields=["index","memory.used","utilization.gpu"]
        rows=nvidia_smi_query(fields)
    data=[]
    for row in rows or []:
        data.append({fields[i].replace(".","_"): row[i] for i in range(min(len(fields),len(row)))})
    return {"epoch":time.time(),"fields":fields,"gpus":data if rows is not None else None}

def gpu_compute_pids():
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=20)
    except (OSError, subprocess.TimeoutExpired):
        return None
    if out.returncode != 0:
        return None
    pids = []
    for line in out.stdout.strip().splitlines():
        parts = [c.strip() for c in line.split(",")]
        if parts and parts[0].isdigit():
            pids.append((int(parts[0]), parts[1] if len(parts) > 1 else None))
    return pids


def gpus_are_free(threshold_mib=1500):
    rows = nvidia_smi_query(["index", "memory.used"])
    if rows is None:
        return None
    try:
        return all(float(r[1]) < threshold_mib for r in rows)
    except (ValueError, IndexError):
        return None

def kill_process_tree(proc):
    """SIGKILL the worker's whole process group (posix) or the worker (Windows)."""
    try:
        if os.name == "nt":
            proc.kill()
        else:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
    except (ProcessLookupError, PermissionError, OSError):
        try:
            proc.kill()
        except Exception:
            pass

KNOWN_LEFTOVER_PIDS = set()   # v1.2.4: PIDs that survived a reclaim are never treated as pre-existing baseline later

def reclaim_gpus(baseline_pids=None, max_wait_sec=120.0):
    """Kill only CUDA processes created after this config started; never kill pre-existing notebook/session jobs."""
    info = {"baseline_pids": [], "leftover_pids_killed": [], "free_after_sec": None, "free": None, "smi_unknown": False}
    if not USE_MODEL:
        info["free"] = True; info["free_after_sec"] = 0.0; return info
    baseline = set(int(pid) for pid, _ in (baseline_pids or [])) - KNOWN_LEFTOVER_PIDS
    info["baseline_pids"] = sorted(baseline)
    t0 = time.perf_counter()
    while True:
        pids = gpu_compute_pids()
        if pids is None:
            if USE_MODEL:
                info["smi_unknown"] = True; info["free"] = False; info["free_after_sec"] = time.perf_counter()-t0
                return info
            info["free"] = True; info["free_after_sec"] = time.perf_counter()-t0
            return info
        for pid, mem in pids:
            if pid != os.getpid() and pid not in baseline:
                try:
                    os.kill(pid, signal.SIGKILL if os.name != "nt" else signal.SIGTERM)
                    info["leftover_pids_killed"].append({"pid": pid, "used_mib": mem})
                except (ProcessLookupError, PermissionError, OSError):
                    pass
        state = gpus_are_free()
        # If pre-existing baseline CUDA work exists, 'globally free' is not expected; only require no new PIDs.
        now_pids = gpu_compute_pids()
        new_left = [] if now_pids is None else [pid for pid, _ in now_pids if pid not in baseline and pid != os.getpid()]
        if not new_left and (state is True or baseline):
            info["free"] = True; info["free_after_sec"] = time.perf_counter()-t0
            return info
        if time.perf_counter() - t0 > max_wait_sec:
            info["free"] = False; info["free_after_sec"] = time.perf_counter()-t0
            KNOWN_LEFTOVER_PIDS.update(new_left)
            info["known_leftover_pids"] = sorted(KNOWN_LEFTOVER_PIDS)
            return info
        time.sleep(3.0)

LOG_PATTERNS = {
    "kv_cache_tokens":        re.compile(r"GPU KV cache size:\s*([\d,]+)\s*tokens"),
    "max_concurrency_x":      re.compile(r"Maximum concurrency for\s*([\d,]+)\s*tokens per request:\s*([\d.]+)x"),
    "graph_capture_sec":      re.compile(r"Graph capturing finished in\s*([\d.]+)\s*secs"),
    "torch_compile_total_sec": re.compile(r"torch\.compile takes\s*([\d.]+)\s*s in total"),
    "model_loading":          re.compile(r"Model loading took\s*([\d.]+)\s*GiB memory and\s*([\d.]+)\s*seconds"),
}
ERROR_HINT = re.compile(r"(CUDA out of memory|OutOfMemory|torch\.OutOfMemoryError|RuntimeError|ValueError|NotImplementedError|Error:)")
CACHE_HINT = re.compile(r"(prefix caching|prefix cache|mamba|Mamba|cudagraph|CUDA graph|custom allreduce|custom all-reduce|Custom allreduce|kv_cache_dtype|KV cache dtype|speculative|MTP|async scheduling|dual batch|DBO|chunked prefill)", re.IGNORECASE)
PREEMPT_HINT = re.compile(r"preempt", re.IGNORECASE)

def parse_worker_log(log_path):
    facts = {"kv_cache_tokens": None, "max_concurrency_x": None, "max_concurrency_len": None, "graph_capture_sec": None,
             "torch_compile_total_sec": None, "model_loading_gib": None, "model_loading_sec": None, "error_hints": [],
             "engine_feature_lines": [], "preemption_lines": [], "preemption_line_count": 0, "log_lines": 0}
    if not Path(log_path).exists():
        return facts
    with open(log_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            facts["log_lines"] += 1
            m = LOG_PATTERNS["kv_cache_tokens"].search(line)
            if m: facts["kv_cache_tokens"] = int(m.group(1).replace(",", ""))
            m = LOG_PATTERNS["max_concurrency_x"].search(line)
            if m: facts["max_concurrency_len"], facts["max_concurrency_x"] = int(m.group(1).replace(",", "")), float(m.group(2))
            m = LOG_PATTERNS["graph_capture_sec"].search(line)
            if m: facts["graph_capture_sec"] = float(m.group(1))
            m = LOG_PATTERNS["torch_compile_total_sec"].search(line)
            if m: facts["torch_compile_total_sec"] = float(m.group(1))
            m = LOG_PATTERNS["model_loading"].search(line)
            if m: facts["model_loading_gib"], facts["model_loading_sec"] = float(m.group(1)), float(m.group(2))
            if ERROR_HINT.search(line) and len(facts["error_hints"]) < 12:
                facts["error_hints"].append(line.rstrip()[:300])
            if CACHE_HINT.search(line) and len(facts["engine_feature_lines"]) < 40:
                facts["engine_feature_lines"].append(line.rstrip()[:300])
            if PREEMPT_HINT.search(line):
                facts["preemption_line_count"] += 1
                if len(facts["preemption_lines"]) < 20: facts["preemption_lines"].append(line.rstrip()[:500])
    return facts

def build_spec(cfg):
    engine_kwargs = dict(Q1_FROZEN_ENGINE)
    engine_kwargs["model"] = str(MODEL_PATH)
    engine_kwargs.update(cfg["overrides"])
    return {
        "name": cfg["name"], "hypothesis": cfg.get("hypothesis"), "optional": bool(cfg.get("optional", False)),
        "engine_kwargs": engine_kwargs, "overrides": dict(cfg["overrides"]),
        "decode_points": list(cfg["decode_points"]), "output_tokens": OUTPUT_TOKENS, "seed": SEED,
        "prompts_path": str(PROMPTS_PATH), "warmup_prompts": WARMUP_PROMPTS, "prefill_prompts": PREFILL_PROMPTS,
        "fingerprint_prompts": FINGERPRINT_PROMPTS, "fingerprint_tokens": FINGERPRINT_TOKENS,
        "fake_time_scale": 1000.0 if FAKE_ENGINE else 1.0,
        "timeout_sec": float(cfg.get("timeout_sec", PER_CONFIG_TIMEOUT_SEC)),
        "visible_devices": str(cfg.get("visible_devices", "0,1,2,3")),
        "anchor_long": bool(cfg.get("anchor_long", False)),
        "anchor_output_tokens": int(ANCHOR_OUTPUT_TOKENS), "anchor_batch_n": int(ANCHOR_BATCH_N),
        "solver_long_n": cfg.get("solver_long_n"), "solver_long_tokens": int(SOLVER_LONG_OUTPUT_TOKENS),
        "mixed_batch_n": int(MIXED_BATCH_N), "mixed_output_fractions": list(MIXED_OUTPUT_FRACTIONS),
        "stability_batch_n": 32, "stability_tokens": int(STABILITY_TOKENS), "stability_repeats": int(STABILITY_REPEATS),
        "agent_turn_n": cfg.get("agent_turn_n"), "agent_turn_output_tokens": int(AGENT_TURN_OUTPUT_TOKENS),
        "drift_sentinel": bool(cfg.get("drift_sentinel", False)),
        "source_frozen": bool(cfg.get("source_frozen", False)),
        "require_request_metrics": bool(cfg.get("require_request_metrics", not cfg.get("source_frozen", False) and not cfg.get("drift_sentinel", False))),
        "cache_ab_repeats": int(CACHE_AB_REPEATS),
        "experimental_no_solver_promotion": bool(cfg.get("experimental_no_solver_promotion", False)),
    }

def run_config(cfg):
    spec = build_spec(cfg)
    name = cfg["name"]
    spec_path = WORK_ROOT / "specs" / f"{name}.json"
    out_path = WORK_ROOT / "results" / f"{name}.json"
    log_path = WORK_ROOT / "logs" / f"{name}.log"
    samples_path = WORK_ROOT / "gpu_samples" / f"{name}.jsonl"
    write_json(spec_path, spec)
    env = dict(os.environ)
    env["VLLM_LOGGING_LEVEL"] = "INFO"
    env["CUDA_VISIBLE_DEVICES"] = str(spec.get("visible_devices", "0,1,2,3"))
    baseline_compute_pids = gpu_compute_pids()
    cmd = [sys.executable, str(WORKER_PATH), "--spec", str(spec_path), "--out", str(out_path)] + (["--fake"] if FAKE_ENGINE else [])
    popen_kwargs = {}
    if os.name == "nt":
        popen_kwargs["creationflags"] = getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0)
    else:
        popen_kwargs["start_new_session"] = True
    record = {"config": name, "spec": spec, "launched_epoch": time.time(), "status": None, "timed_out": False,
              "returncode": None, "parent_wall_sec": None, "result": None, "log_facts": None, "gpu_reclaim": None}
    t0 = time.perf_counter()
    last_sample = 0.0
    with open(log_path, "w", encoding="utf-8") as logf, open(samples_path, "w", encoding="utf-8") as sf:
        proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env, cwd=str(WORK_ROOT), **popen_kwargs)
        while True:
            rc = proc.poll()
            now = time.perf_counter()
            if now - last_sample >= GPU_SAMPLE_INTERVAL_SEC:
                s = gpu_sample()
                if s["gpus"] is not None:
                    sf.write(json.dumps(s) + "\n"); sf.flush()
                last_sample = now
            if rc is not None:
                break
            if now - t0 > spec["timeout_sec"]:
                kill_process_tree(proc)
                try:
                    proc.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    pass
                record["timed_out"] = True
                rc = proc.returncode
                break
            time.sleep(0.1 if FAKE_ENGINE else 1.0)
    record["returncode"] = rc
    record["parent_wall_sec"] = time.perf_counter() - t0
    record["gpu_reclaim"] = reclaim_gpus(baseline_compute_pids)
    if out_path.exists():
        with open(out_path, "r", encoding="utf-8") as f:
            record["result"] = json.load(f)
    record["log_facts"] = parse_worker_log(log_path)
    res_status = (record["result"] or {}).get("status")
    if record["timed_out"]:
        record["status"] = "timeout"
    elif res_status in ("ok", "unsupported", "failed"):
        record["status"] = res_status
    elif record["result"] is None:
        record["status"] = "no_result"
    else:
        record["status"] = "died_mid_run"   # worker started/partial phases but never wrote a terminal status
    write_json(WORK_ROOT / "results" / f"{name}.record.json", record)
    return record

MEASURED_PREFILL_TPS = None   # v1.2.4: updated from each successful config's cold prefill wave

def measured_speed_prior(records):
    """(est_load_sec, est_sec_per_output_token) from the latest successful config, else priors."""
    global MEASURED_PREFILL_TPS
    load, spt = PRIOR_LOAD_SEC, PRIOR_SEC_PER_OUTPUT_TOKEN
    for rec in records:
        r = rec.get("result") or {}
        if rec.get("status") == "ok":
            if r.get("load_sec"):
                load = float(r["load_sec"]) + 30.0
            dec = [p for p in r.get("phases", []) if str(p.get("phase", "")).startswith("decode@") and p.get("sec_per_output_token_per_seq")]
            if dec:
                spt = min(float(p["sec_per_output_token_per_seq"]) for p in dec)
            cold = next((p for p in r.get("phases", []) if p.get("phase") == "prefill_cold" and p.get("effective_full_prefill_tps")), None)
            if cold:
                MEASURED_PREFILL_TPS = float(cold["effective_full_prefill_tps"])
    return load, spt

def project_config_sec(cfg, est_load, est_spt):
    if cfg.get("projected_sec_override") is not None:      # selftest hook
        return float(cfg["projected_sec_override"])
    mns=int(cfg["overrides"].get("max_num_seqs",Q1_FROZEN_ENGINE["max_num_seqs"]))
    pts=[int(n) for n in cfg["decode_points"] if int(n)<=mns]
    decode=sum(OUTPUT_TOKENS*est_spt*(1.0+0.01*n) for n in pts)
    if cfg.get("drift_sentinel"): return est_load+decode+90.0
    mixed=OUTPUT_TOKENS*est_spt*0.75; stability=STABILITY_REPEATS*STABILITY_TOKENS*est_spt
    solver_long=(SOLVER_LONG_OUTPUT_TOKENS*est_spt*(1.0+0.01*int(cfg.get("solver_long_n") or 0))) if cfg.get("solver_long_n") else 0.0
    agent=(AGENT_TURN_OUTPUT_TOKENS*est_spt*(1.0+0.01*int(cfg.get("agent_turn_n") or 0))) if cfg.get("agent_turn_n") else 0.0
    # v1.2.4: the long anchor and the prefill waves were missing from the projection.
    anchor=(ANCHOR_OUTPUT_TOKENS*est_spt*(1.0+0.01*ANCHOR_BATCH_N)) if cfg.get("anchor_long") else 0.0
    k=min(PREFILL_PROMPTS, len(PROMPT_POOL), mns)
    waves=(2 + CACHE_AB_REPEATS*3 + 4 + (2 if cfg.get("agent_turn_n") else 0)) if cfg["overrides"].get("enable_prefix_caching") else 2
    med=float(PROMPT_POOL_STATS.get("prompt_tokens_median") or 6000.0)
    prefill=(waves*k*(med+1500.0))/float(MEASURED_PREFILL_TPS or PRIOR_PREFILL_TPS)
    return est_load+decode+mixed+stability+solver_long+agent+anchor+prefill+CONFIG_OVERHEAD_SEC



def run_tp2x2_pair(disable_custom_all_reduce,pairing=("0,1","2,3"),pairing_tag="p01_23",absolute_deadline=None):
    """Measure two TP2 replicas concurrently on a chosen perfect matching of the four GPUs."""
    mode="noAR" if disable_custom_all_reduce else "AR"; name=f"T3_tp2x2_pair_{mode}_{pairing_tag}"
    barrier = WORK_ROOT / "pair_barrier" / name
    if barrier.exists(): shutil.rmtree(barrier)
    barrier.mkdir(parents=True, exist_ok=True)
    # GPU telemetry/reclaim are meaningful only in real-GPU profiles.  In SELFTEST_CPU,
    # bypass nvidia-smi entirely so the synchronization path is tested without depending
    # on host GPU tooling or stale CUDA processes from unrelated sessions.
    baseline = gpu_compute_pids() if USE_MODEL else []
    pair_samples_path = WORK_ROOT / "gpu_samples" / f"{name}.jsonl"
    member_meta, procs = [], []
    for mid, devices in enumerate(pairing):
        cfg = {"name": f"{name}_m{mid}", "overrides": dict(_GRAPH_NOAR if disable_custom_all_reduce else _GRAPH_AR,
                tensor_parallel_size=2, max_num_seqs=64), "decode_points": [TP2_PAIR_N], "visible_devices": devices}
        spec = build_spec(cfg)
        spec.update({"pair_member": True, "pair_member_id": mid, "pair_n": TP2_PAIR_N,
                     "pair_prompt_indices":list(TP2_PAIR_PROMPT_PLAN["replica_indices"][str(mid)]),
                     "pair_prompt_task_ids":list(TP2_PAIR_PROMPT_PLAN["replica_task_ids"][str(mid)]),
                     "barrier_dir": str(barrier), "barrier_timeout_sec": TP2_PAIR_READY_MAX_SEC,
                     "timeout_sec": TP2_PAIR_TIMEOUT_SEC})
        spath = WORK_ROOT / "specs" / f"{cfg['name']}.json"
        opath = WORK_ROOT / "results" / f"{cfg['name']}.json"
        lpath = WORK_ROOT / "logs" / f"{cfg['name']}.log"
        write_json(spath, spec)
        env = dict(os.environ); env["VLLM_LOGGING_LEVEL"]="INFO"; env["CUDA_VISIBLE_DEVICES"] = devices
        logf = open(lpath, "w", encoding="utf-8")
        kw = {"start_new_session": True} if os.name != "nt" else {"creationflags": getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0)}
        p = subprocess.Popen([sys.executable, str(WORKER_PATH), "--spec", str(spath), "--out", str(opath)] + (["--fake"] if FAKE_ENGINE else []),
                             stdout=logf, stderr=subprocess.STDOUT, env=env, cwd=str(WORK_ROOT), **kw)
        procs.append((p, logf)); member_meta.append((cfg, spec, opath, lpath))
    t0=time.perf_counter(); ready=False; _last_sample=0.0
    hard_deadline=min(t0+TP2_PAIR_TIMEOUT_SEC, float(absolute_deadline) if absolute_deadline is not None else t0+TP2_PAIR_TIMEOUT_SEC)
    ready_deadline=min(t0+TP2_PAIR_READY_MAX_SEC, hard_deadline)
    with open(pair_samples_path,"w",encoding="utf-8") as _psf:
        while time.perf_counter() < ready_deadline:
            _now=time.perf_counter()
            if USE_MODEL and _now-_last_sample>=GPU_SAMPLE_INTERVAL_SEC:
                _gs=gpu_sample();
                if _gs.get("gpus") is not None: _psf.write(json.dumps(_gs)+"\n"); _psf.flush()
                _last_sample=_now
            if all((barrier/f"ready_{i}").exists() for i in (0,1)):
                ready=True; break
            if any(p.poll() is not None for p,_ in procs): break
            time.sleep(0.1 if FAKE_ENGINE else 1.0)
    if ready: (barrier/"GO").write_text("go")
    timed_out=False
    with open(pair_samples_path,"a",encoding="utf-8") as _psf:
        while any(p.poll() is None for p,_ in procs):
            _now=time.perf_counter()
            if USE_MODEL and _now-_last_sample>=GPU_SAMPLE_INTERVAL_SEC:
                _gs=gpu_sample();
                if _gs.get("gpus") is not None: _psf.write(json.dumps(_gs)+"\n"); _psf.flush()
                _last_sample=_now
            if _now>hard_deadline:
                timed_out=True
                for p,_ in procs:
                    if p.poll() is None: kill_process_tree(p)
                break
            time.sleep(0.1 if FAKE_ENGINE else 1.0)
    for p,lf in procs:
        try: p.wait(timeout=20)
        except Exception: pass
        lf.close()
    reclaim = reclaim_gpus(baseline) if USE_MODEL else {
        "baseline_pids": [], "leftover_pids_killed": [], "free_after_sec": 0.0,
        "free": True, "smi_unknown": False, "selftest_gpu_bypass": True,
    }
    members=[]
    for cfg,spec,opath,lpath in member_meta:
        data=json.loads(opath.read_text()) if opath.exists() else None
        members.append({"config": cfg["name"], "status": (data or {}).get("status"), "result": data,
                        "log_facts": parse_worker_log(lpath), "spec": spec})
    good = ready and not timed_out and all(m["status"]=="ok" for m in members)
    pair_phase=None
    if good:
        phases=[]
        for m in members:
            phases.extend([p for p in (m["result"] or {}).get("phases",[]) if str(p.get("phase","")).startswith("pair_decode@")])
        if len(phases)==2:
            wall=max(float(p["wall_sec"]) for p in phases)
            out_total=sum(int(p["out_tokens_total"]) for p in phases)
            pair_phase={"phase": f"pair_decode@{TP2_PAIR_N}x2", "n": TP2_PAIR_N*2, "wall_sec": wall,
                        "out_tokens_total": out_total, "out_tokens_min": min(p["out_tokens_min"] for p in phases),
                        "out_tokens_max": max(p["out_tokens_max"] for p in phases),
                        "all_reached_max_tokens": all(p["all_reached_max_tokens"] for p in phases),
                        "agg_gen_tps": out_total/wall if wall>0 else None,
                        "per_seq_gen_tps": (out_total/(TP2_PAIR_N*2))/wall if wall>0 else None,
                        "sec_per_output_token_per_seq": wall/max(1, OUTPUT_TOKENS),
                        "prompt_tokens_total_tokenizer": sum(p.get("prompt_tokens_total_tokenizer",0) for p in phases),
                        "prompt_tokens_total_engine": None,
                        "prompt_indices":sum([list(p.get("prompt_indices") or []) for p in phases],[]),
                        "prompt_task_ids":sum([list(p.get("prompt_task_ids") or []) for p in phases],[]),
                        "replica_prompt_overlap_count":len(set(phases[0].get("prompt_indices") or []) & set(phases[1].get("prompt_indices") or [])),
                        "prompts_repeated":len(set(sum([list(p.get("prompt_indices") or []) for p in phases],[]))) < TP2_PAIR_N*2,
                        "phase_integrity":all(bool(p.get("phase_integrity")) for p in phases),
                        "note": "two synchronized TP2 replicas on GPU pairing %s"%("|".join(pairing))}
            if pair_phase["replica_prompt_overlap_count"] != 0 or pair_phase["prompts_repeated"] or not pair_phase["phase_integrity"]:
                raise RuntimeError(f"TP2x2 aggregate integrity failure: overlap={pair_phase['replica_prompt_overlap_count']} repeated={pair_phase['prompts_repeated']} integrity={pair_phase['phase_integrity']}")
    rec={"config": name, "status": "ok" if pair_phase else ("timeout" if timed_out else "failed"),
         "timed_out": timed_out, "returncode": None, "parent_wall_sec": time.perf_counter()-t0,
         "spec": {"name":name, "optional":True, "overrides":{"tensor_parallel_size":2,"replicas":2,
                  "disable_custom_all_reduce":disable_custom_all_reduce}, "visible_devices":"|".join(pairing), "pairing":list(pairing), "pairing_tag":pairing_tag},
         "result": {"status":"ok" if pair_phase else "failed", "phases":[pair_phase] if pair_phase else [],
                    "replica_results":members, "engine_kwargs_used":None, "error":None if pair_phase else "pair did not complete"},
         "log_facts": {}, "gpu_reclaim": reclaim}
    write_json(WORK_ROOT/"results"/f"{name}.record.json", rec)
    return rec

_gpu_free_boot = gpus_are_free()
if USE_MODEL and _gpu_free_boot is not True:
    raise RuntimeError(f"GPU profile requires a provably clean GPU state at boot; gpus_are_free={_gpu_free_boot}, compute_pids={gpu_compute_pids()}")
print("parent helpers ready; GPUs free now:", _gpu_free_boot, "| compute pids:", gpu_compute_pids())


worker script: /kaggle/working/tp_q27_probe_v124/smoke_gpu/20260916T091344Z/tp_worker.py sha256 4a4d54b61fe2a3d0 compile OK
parent helpers ready; GPUs free now: True | compute pids: []


In [5]:
# =====================================================================================
# CELL 5 — RUN THE LADDER (one subprocess per config; deadline-projected; failures recorded)
# =====================================================================================
RECORDS = []
LADDER_LOG = []
GPU_STATE_COMPROMISED = False
GPU_STATE_COMPROMISED_REASON = None

def _fmt_tps(x):
    return "-" if x is None else f"{x:7.1f}"

def print_config_row(rec):
    r = rec.get("result") or {}
    dec = {p["phase"]: p for p in r.get("phases", []) if str(p.get("phase", "")).startswith("decode@") and not p.get("skipped")}
    parts = [f"{rec['config']:<24} {rec['status']:<12} load={_fmt_tps(r.get('load_sec'))}s"]
    for ph, p in sorted(dec.items(), key=lambda kv: kv[1]["n"]):
        parts.append(f"{ph}: agg={_fmt_tps(p['agg_gen_tps'])} tok/s per_seq={p['per_seq_gen_tps']:.2f}")
    lf = rec.get("log_facts") or {}
    if lf.get("kv_cache_tokens"):
        parts.append(f"KV={lf['kv_cache_tokens']:,} tok ({lf.get('max_concurrency_x')}x @ {lf.get('max_concurrency_len')})")
    if rec["status"] not in ("ok",):
        err = (r.get("error") or "")[:160]
        hints = (lf.get("error_hints") or [])[-2:]
        parts.append(f"error={err!r} hints={hints}")
    print("  " + " | ".join(parts), flush=True)

_required_remaining = [c for c in CONFIG_LADDER if not c.get("optional")]
_SENTINELS = [c for c in CONFIG_LADDER if c.get("drift_sentinel")]
_MAIN_LADDER = [c for c in CONFIG_LADDER if not c.get("drift_sentinel")]

def _run_or_skip(cfg, i, total):
    global GPU_STATE_COMPROMISED, GPU_STATE_COMPROMISED_REASON
    if USE_MODEL and GPU_STATE_COMPROMISED:
        rec={"config":cfg["name"],"spec":build_spec(cfg),"status":"skipped_dirty_gpu","timed_out":False,"returncode":None,"parent_wall_sec":0.0,
             "result":None,"log_facts":None,"gpu_reclaim":None,"skip_reason":GPU_STATE_COMPROMISED_REASON}
        write_json(WORK_ROOT/"results"/f"{cfg['name']}.record.json",rec); RECORDS.append(rec)
        print(f"[{i+1}/{total}] {cfg['name']}: SKIPPED dirty GPU ({GPU_STATE_COMPROMISED_REASON})",flush=True); return rec
    est_load, est_spt = measured_speed_prior(RECORDS)
    proj = project_config_sec(cfg, est_load, est_spt)
    # Always reserve the end-of-session drift sentinel; without it the cross-config ranking
    # is scientifically invalid. Optional rows additionally reserve every remaining required row.
    _sentinel_reserve = sum(project_config_sec(c, est_load, est_spt) for c in _SENTINELS if c["name"] != cfg["name"])
    _other_required_reserve = sum(project_config_sec(c, est_load, est_spt) for c in _required_remaining
                                  if c["name"] != cfg["name"] and not c.get("drift_sentinel")) if cfg.get("optional") else 0.0
    reserve_for_required = _sentinel_reserve + _other_required_reserve
    now = time.perf_counter()
    remaining = DEADLINE - now
    entry = {"config": cfg["name"], "index": i, "projected_sec": proj, "remaining_sec": remaining,
             "reserve_for_required_sec": reserve_for_required, "optional": bool(cfg.get("optional"))}
    if proj + reserve_for_required > remaining:
        entry["decision"] = "skipped_deadline"
        LADDER_LOG.append(entry)
        rec = {"config": cfg["name"], "spec": build_spec(cfg), "status": "skipped_deadline", "timed_out": False,
               "returncode": None, "parent_wall_sec": 0.0, "result": None, "log_facts": None, "gpu_reclaim": None,
               "skip_reason": f"projected {proj/60:.1f} min (+{reserve_for_required/60:.1f} min reserved for required configs) > remaining {remaining/60:.1f} min"}
        write_json(WORK_ROOT / "results" / f"{cfg['name']}.record.json", rec)
        RECORDS.append(rec)
        print(f"[{i+1}/{total}] {cfg['name']}: SKIPPED ({rec['skip_reason']})", flush=True)
        return rec
    entry["decision"] = "run"
    LADDER_LOG.append(entry)
    print(f"[{i+1}/{total}] {cfg['name']}  projected {proj/60:.1f} min | remaining {remaining/60:.1f} min | overrides={cfg['overrides']}", flush=True)
    print(f"    hypothesis: {cfg.get('hypothesis')}", flush=True)
    rec = run_config(cfg)
    RECORDS.append(rec)
    print_config_row(rec)
    if USE_MODEL and (not rec.get("gpu_reclaim") or rec["gpu_reclaim"].get("free") is not True):
        GPU_STATE_COMPROMISED=True
        GPU_STATE_COMPROMISED_REASON=f"GPU reclaim failed after {cfg['name']}: {rec.get('gpu_reclaim')}"
        print("    FATAL FOR COMPARISON:",GPU_STATE_COMPROMISED_REASON,flush=True)
    return rec

for i, cfg in enumerate(_MAIN_LADDER):
    if not cfg.get("optional"):
        _required_remaining = [c for c in _required_remaining if c["name"] != cfg["name"]]
    _run_or_skip(cfg, i, len(CONFIG_LADDER))


def _pair_projected_sec():
    est_load,est_spt=measured_speed_prior(RECORDS)
    return min(TP2_PAIR_TIMEOUT_SEC, est_load + OUTPUT_TOKENS*est_spt*(1.0+0.01*TP2_PAIR_N) + CONFIG_OVERHEAD_SEC)

def _run_pair_if_admitted(disable_ar,pairing,tag,label):
    global GPU_STATE_COMPROMISED, GPU_STATE_COMPROMISED_REASON
    if USE_MODEL and GPU_STATE_COMPROMISED:
        print(f"{label} skipped: dirty GPU state: {GPU_STATE_COMPROMISED_REASON}"); return None
    _sentinel_reserve=sum(project_config_sec(c,*measured_speed_prior(RECORDS)) for c in _SENTINELS)
    _pair_cost=_pair_projected_sec(); _remaining=DEADLINE-time.perf_counter()
    if not FAKE_ENGINE and _pair_cost+_sentinel_reserve>_remaining:
        print(f"{label} skipped: pair projection {_pair_cost/60:.1f}m + sentinel {_sentinel_reserve/60:.1f}m > remaining {_remaining/60:.1f}m"); return None
    _absolute=DEADLINE-_sentinel_reserve if not FAKE_ENGINE else time.perf_counter()+TP2_PAIR_TIMEOUT_SEC
    _tr=run_tp2x2_pair(disable_ar,pairing,tag,absolute_deadline=_absolute)
    TOPOLOGY_RECORDS.append(_tr); print_config_row(_tr)
    if USE_MODEL and (not _tr.get("gpu_reclaim") or _tr["gpu_reclaim"].get("free") is not True):
        GPU_STATE_COMPROMISED=True; GPU_STATE_COMPROMISED_REASON=f"GPU reclaim failed after {_tr['config']}: {_tr.get('gpu_reclaim')}"
    return _tr

TOPOLOGY_RECORDS=[]
_tp2_status={r["config"]:r["status"] for r in RECORDS if r["config"] in ("T1_tp2_noAR","T2_tp2_AR")}
_default_pair=("0,1","2,3")
for _disable_ar,_single_name in [(True,"T1_tp2_noAR"),(False,"T2_tp2_AR")]:
    if FAKE_ENGINE or _tp2_status.get(_single_name)=="ok":
        print(f"Synchronized TP2x2 default-pair test from {_single_name} (disable_custom_all_reduce={_disable_ar})")
        _run_pair_if_admitted(_disable_ar,_default_pair,"p01_23",f"TP2x2 {_single_name} default-pair")
    else: print(f"TP2x2 {_single_name} default-pair skipped: single row unavailable")

# Full probe: hold AR mode fixed at the faster single-TP2 choice and sweep the other perfect matchings.
# v1.2.4: ONE code path for PROBE_GPU and SELFTEST_CPU (v1.2.1 had a `{{ }}` set-of-dict TypeError here that
# only the un-selftested PROBE_GPU branch executed).
if (RUN_PROFILE=="PROBE_GPU" and USE_MODEL) or FAKE_ENGINE:
    _singles={r["config"]:r for r in RECORDS if r["config"] in ("T1_tp2_noAR","T2_tp2_AR") and r["status"]=="ok"}
    def _best_single_tps(rec):
        vals=[p.get("agg_gen_tps") or 0 for p in (rec.get("result") or {}).get("phases",[]) if str(p.get("phase"," ")).startswith("decode@") and not p.get("skipped")]
        return max(vals or [0])
    if _singles:
        _winner=max(_singles.values(),key=_best_single_tps); _disable=("noAR" in _winner["config"])
        for _pair,_tag in [(("0,2","1,3"),"p02_13"),(("0,3","1,2"),"p03_12")]:
            print(f"Topology pairing sweep {_tag}; mode={'noAR' if _disable else 'AR'}")
            _run_pair_if_admitted(_disable,_pair,_tag,f"Topology pairing {_tag}")
    else:
        print("Topology pairing sweep skipped: no successful single TP2 row")

# v1.2.4: the frozen-Q1 drift sentinel runs LAST, after every topology test, so it brackets the whole session.
for _j, _cfg in enumerate(_SENTINELS):
    _run_or_skip(_cfg, len(_MAIN_LADDER) + _j, len(CONFIG_LADDER))

write_json(WORK_ROOT / "analysis" / "ladder_log.json", LADDER_LOG)
print("\nLadder finished in", f"{elapsed_sec()/60:.1f} min;", {r["status"] for r in RECORDS}, "| topology", {r["config"]:r["status"] for r in TOPOLOGY_RECORDS})


[1/8] C0_frozen_q1  projected 28.3 min | remaining 358.0 min | overrides={}
    hypothesis: Frozen Q1 anchor. Dedicated long decode@8 must reproduce the prior ~0.16 s/token/sequence within tolerance; request stats remain source-faithful/off.
  C0_frozen_q1             ok           load=  281.1s | decode@8: agg=   27.1 tok/s per_seq=3.39 | decode@16: agg=   37.7 tok/s per_seq=2.36 | KV=159,936 tok (19.9x @ 29696)
[2/8] I0_q1_stats  projected 16.8 min | remaining 330.4 min | overrides={'disable_log_stats': False}
    hypothesis: Instrumentation control: frozen Q1 engine + only disable_log_stats=False. Optimization rows compare against this when request-level latency is needed.
  I0_q1_stats              ok           load=  208.8s | decode@8: agg=   24.6 tok/s per_seq=3.07 | decode@16: agg=   37.7 tok/s per_seq=2.36 | KV=159,936 tok (19.9x @ 29696)
[3/8] C1_graph_noAR  projected 34.1 min | remaining 315.4 min | overrides={'enforce_eager': False, 'disable_custom_all_reduce': True, 'disable

In [6]:
# =====================================================================================
# CELL 6 — COMPREHENSIVE ANALYSIS / GATES / FLAT MEASUREMENT FILES
# =====================================================================================
import pandas as pd
import statistics

ANALYSIS_RECORDS = list(RECORDS) + list(TOPOLOGY_RECORDS)

def _decode_phases(rec):
    r=rec.get("result") or {}
    return sorted([p for p in r.get("phases",[]) if str(p.get("phase","")).startswith(("decode@","pair_decode@")) and not p.get("skipped")], key=lambda p:p.get("n",0))
def _phase(rec,name):
    for p in (rec.get("result") or {}).get("phases",[]):
        if p.get("phase")==name and not p.get("skipped"): return p
    return None

def _load_gpu_samples(name):
    path=WORK_ROOT/"gpu_samples"/f"{name}.jsonl"; out=[]
    if not path.exists(): return out
    for line in path.read_text(encoding="utf-8",errors="replace").splitlines():
        try: out.append(json.loads(line))
        except json.JSONDecodeError: pass
    return out

def _to_float(x):
    try: return float(str(x).strip())
    except Exception: return None

def _gpu_health(name):
    samples=_load_gpu_samples(name); mem=[]; power=[]; temp=[]; smclk=[]; util=[]; pstate=[]
    for s in samples:
        for g in s.get("gpus") or []:
            if isinstance(g,dict):
                for arr,key in [(mem,"memory_used"),(power,"power_draw"),(temp,"temperature_gpu"),(smclk,"clocks_sm"),(util,"utilization_gpu")]:
                    v=_to_float(g.get(key));
                    if v is not None: arr.append(v)
                if g.get("pstate") is not None: pstate.append(str(g.get("pstate")))
    sample_totals=[]
    for sm in samples:
        vals=[]
        for g in sm.get("gpus") or []:
            if isinstance(g,dict):
                v=_to_float(g.get("power_draw"));
                if v is not None: vals.append(v)
        if vals: sample_totals.append(sum(vals))
    return {"samples":len(samples), "peak_gpu_mib":max(mem) if mem else None, "mean_total_power_w":statistics.mean(sample_totals) if sample_totals else None,
            "peak_temp_c":max(temp) if temp else None, "min_sm_clock_mhz":min(smclk) if smclk else None,
            "mean_gpu_util_pct":statistics.mean(util) if util else None, "pstates":sorted(set(pstate))}

def _phase_intervals(rec):
    tl=(rec.get("result") or {}).get("timeline",[]); stacks={}; intervals=[]
    for e in tl:
        ev=str(e.get("event","")); t=e.get("epoch")
        if ev.startswith("phase_start:"): stacks.setdefault(ev.split(":",1)[1],[]).append(t)
        elif ev.startswith("phase_end:"):
            n=ev.split(":",1)[1]; starts=stacks.get(n) or []
            if starts: intervals.append((n,starts.pop(0),t))
    return intervals

def _energy_for_phase(rec, phase_name):
    # Approximate joules = mean sampled total-GPU power during phase * phase duration.
    intervals=[x for x in _phase_intervals(rec) if x[0]==phase_name]
    if not intervals: return None
    start,end=intervals[0][1],intervals[0][2]
    totals=[]
    for s in _load_gpu_samples(rec["config"]):
        if start <= float(s.get("epoch",0)) <= end:
            vals=[]
            for g in s.get("gpus") or []:
                if isinstance(g,dict):
                    v=_to_float(g.get("power_draw"));
                    if v is not None: vals.append(v)
            if vals: totals.append(sum(vals))
    if not totals: return None
    return statistics.mean(totals)*max(0.0,end-start)

def _fingerprint_agreement(a,b):
    if not a or not b or len(a)!=len(b): return None
    fr=[]; exact=[]
    for x,y in zip(a,b):
        n=min(len(x),len(y)); k=0
        while k<n and x[k]==y[k]: k+=1
        fr.append(k/n if n else 0.0); exact.append(x==y)
    return {"mean_prefix_agreement":float(sum(fr)/len(fr)),"prompts_identical":int(sum(exact)),"prompts":len(exact),"exact":all(exact)}

ANCHOR_NAME="C0_frozen_q1"
_anchor=next((r for r in RECORDS if r["config"]==ANCHOR_NAME and r["status"]=="ok"),None)
_anchor_fp=((_anchor or {}).get("result") or {}).get("fingerprint")
_drift=next((r for r in RECORDS if r["config"]=="Z0_frozen_q1_repeat" and r["status"]=="ok"),None)

phase_rows=[]; rows=[]; stability_rows=[]
for rec in ANALYSIS_RECORDS:
    rr=rec.get("result") or {}; lf=rec.get("log_facts") or {}; dec=_decode_phases(rec)
    cold=_phase(rec,"prefill_cold"); exact=_phase(rec,"prefill_warm_exact")
    ext_small=_phase(rec,"prefill_warm_extension_small"); ext_large=_phase(rec,"prefill_warm_extension_large")
    uncached_ab=[p for p in rr.get("phases",[]) if str(p.get("phase"," ")).startswith("cache_ab_uncached_medium_r")]
    cached_ab=[p for p in rr.get("phases",[]) if str(p.get("phase"," ")).startswith("cache_ab_cached_medium_r")]
    uncached_med=statistics.median([p["wall_sec"] for p in uncached_ab]) if uncached_ab else None
    cached_med=statistics.median([p["wall_sec"] for p in cached_ab]) if cached_ab else None
    ext=(cached_ab[0] if cached_ab else None)
    hist=next((p for p in rr.get("phases",[]) if str(p.get("phase"," ")).startswith("tool_history_seed")),None); agent=next((p for p in rr.get("phases",[]) if str(p.get("phase"," ")).startswith("agent_turn_extension@")),None)
    solver_long=next((p for p in rr.get("phases",[]) if str(p.get("phase"," ")).startswith("solver_long@")),None)
    health=_gpu_health(rec["config"])
    best=max(dec,key=lambda p:p.get("agg_gen_tps") or 0) if dec else None
    st=[p for p in rr.get("phases",[]) if str(p.get("phase","")).startswith("stability@")]
    st_tps=[float(p["agg_gen_tps"]) for p in st if p.get("agg_gen_tps")]
    st_cv=(statistics.pstdev(st_tps)/statistics.mean(st_tps)) if len(st_tps)>=2 and statistics.mean(st_tps)>0 else None
    mixed=next((p for p in rr.get("phases",[]) if str(p.get("phase","")).startswith("mixed_decode@")),None)
    fp=_fingerprint_agreement(_anchor_fp,rr.get("fingerprint")) if (_anchor_fp and rr.get("fingerprint")) else None
    exp_no_prom=bool((rec.get("spec") or {}).get("experimental_no_solver_promotion",False))
    row={"config":rec["config"],"status":rec["status"],"optional":bool((rec.get("spec") or {}).get("optional",False)),
         "experimental_no_solver_promotion":exp_no_prom,
         "overrides":json.dumps((rec.get("spec") or {}).get("overrides",{}),sort_keys=True),
         "load_sec":rr.get("load_sec"),"kv_cache_tokens":lf.get("kv_cache_tokens"),"max_concurrency_x":lf.get("max_concurrency_x"),
         "graph_capture_sec":lf.get("graph_capture_sec"),"torch_compile_sec":lf.get("torch_compile_total_sec"),
         "best_point_n":best.get("n") if best else None,"best_agg_gen_tps":best.get("agg_gen_tps") if best else None,
         "best_per_seq_gen_tps":best.get("per_seq_gen_tps") if best else None,
         "mixed_agg_gen_tps":mixed.get("agg_gen_tps") if mixed else None,"mixed_total_output_tokens":mixed.get("out_tokens_total") if mixed else None,
         "solver_long_agg_gen_tps":solver_long.get("agg_gen_tps") if solver_long else None,"solver_long_n":solver_long.get("n") if solver_long else None,
         "stability_mean_tps":statistics.mean(st_tps) if st_tps else None,"stability_cv":st_cv,
         "prefill_cold_effective_tps":cold.get("effective_full_prefill_tps") if cold else None,
         "phase_integrity_all":all(bool(p.get("phase_integrity",True)) for p in rr.get("phases",[]) if not p.get("skipped")),
         "request_metrics_required":bool((rec.get("spec") or {}).get("require_request_metrics",False)),
         "request_metrics_available":all((p.get("request_timing_available_count",0)==p.get("n")) for p in rr.get("phases",[]) if not p.get("skipped")) if (rec.get("spec") or {}).get("require_request_metrics",False) else None,
         "prefix_exact_wall_speedup":(cold["wall_sec"]/exact["wall_sec"]) if cold and exact and exact.get("wall_sec") else None,
         "prefix_extension_small_wall_speedup":None,
         "prefix_extension_wall_speedup":(uncached_med/cached_med) if uncached_med and cached_med else None,
         "prefix_extension_uncached_wall_median":uncached_med,"prefix_extension_cached_wall_median":cached_med,
         "prefix_extension_ab_repeats":min(len(uncached_ab),len(cached_ab)),
         "prefix_extension_large_wall_speedup":None,
         "prefix_extension_small_suffix_tokens":ext_small.get("new_suffix_tokens_total") if ext_small else None,
         "prefix_extension_new_suffix_tokens":ext.get("new_suffix_tokens_total") if ext else None,
         "prefix_extension_large_suffix_tokens":ext_large.get("new_suffix_tokens_total") if ext_large else None,
         "prefix_extension_small_suffix_tps":ext_small.get("new_suffix_tps") if ext_small else None,
         "prefix_extension_new_suffix_tps":ext.get("new_suffix_tps") if ext else None,
         "prefix_extension_large_suffix_tps":ext_large.get("new_suffix_tps") if ext_large else None,
         "prefix_extension_cache_ratio_expected":ext.get("cached_vs_expected_shared_ratio") if ext else None,
         "prefix_extension_cached_tokens":ext.get("num_cached_tokens_total") if ext else None,
         "agent_turn_agg_gen_tps":agent.get("agg_gen_tps") if agent else None,"agent_turn_cache_ratio_expected":agent.get("cached_vs_expected_shared_ratio") if agent else None,
         "agent_turn_ttft_median":agent.get("ttft_sec_median") if agent else None,"agent_turn_request_decode_tps_median":agent.get("request_decode_tps_median") if agent else None,
         "preemption_line_count":lf.get("preemption_line_count",0),
         "fingerprint_vs_anchor":fp,"numerics_exact":None if fp is None else bool(fp["exact"]),
         "parent_wall_sec":rec.get("parent_wall_sec"),"error":rr.get("error") or rec.get("skip_reason"),
         "unsupported_kwargs":rr.get("unsupported_kwargs"), **health}
    rows.append(row)
    if st_tps: stability_rows.append({"config":rec["config"],"values":st_tps,"mean":statistics.mean(st_tps),"cv":st_cv})
    for p in rr.get("phases",[]):
        if p.get("skipped"): continue
        e=_energy_for_phase(rec,p.get("phase"))
        outtok=p.get("out_tokens_total") or 0
        phase_rows.append({"config":rec["config"],"phase":p.get("phase"),"n":p.get("n"),"wall_sec":p.get("wall_sec"),
                           "agg_gen_tps":p.get("agg_gen_tps"),"out_tokens_total":outtok,"requested_output_tokens_total":p.get("requested_output_tokens_total"),
                           "prompt_tokens_total":p.get("prompt_tokens_total_tokenizer"),"prompt_tokens_median":p.get("prompt_tokens_median"),"prompt_tokens_p90":p.get("prompt_tokens_p90"),
                           "prompt_unique_indices":p.get("prompt_unique_indices"),"energy_j_est":e,"tokens_per_joule":(outtok/e if e and outtok else None),
                           "base_prompt_tokens_total":p.get("base_prompt_tokens_total"),"new_suffix_tokens_total":p.get("new_suffix_tokens_total"),"new_suffix_tps":p.get("new_suffix_tps"),
                           "num_cached_tokens_total":p.get("num_cached_tokens_total"),"cache_hit_ratio_prompt":p.get("cache_hit_ratio_prompt"),
                           "expected_shared_prefix_tokens_total":p.get("expected_shared_prefix_tokens_total"),"cached_vs_expected_shared_ratio":p.get("cached_vs_expected_shared_ratio"),
                           "ttft_sec_median":p.get("ttft_sec_median"),"ttft_sec_p95":p.get("ttft_sec_p95"),"queue_sec_median":p.get("queue_sec_median"),
                           "request_decode_sec_median":p.get("request_decode_sec_median"),"request_decode_sec_p95":p.get("request_decode_sec_p95"),
                           "request_decode_tps_median":p.get("request_decode_tps_median"),"request_decode_tps_p05":p.get("request_decode_tps_p05"),
                           "phase_integrity":p.get("phase_integrity"),"prompt_task_ids":json.dumps(p.get("prompt_task_ids") or []),"prompt_indices":json.dumps(p.get("prompt_indices") or [])})

SUMMARY=pd.DataFrame(rows); SUMMARY.to_csv(WORK_ROOT/"analysis"/"summary_table.csv",index=False)
pd.DataFrame(phase_rows).to_csv(WORK_ROOT/"analysis"/"phase_metrics.csv",index=False)
_common=[r for r in phase_rows if str(r.get("phase"," ")).startswith(("decode@","pair_decode@")) and r.get("n") in (8,16,32,64)]
pd.DataFrame(_common).to_csv(WORK_ROOT/"analysis"/"common_concurrency.csv",index=False)
pd.DataFrame(stability_rows).to_json(WORK_ROOT/"analysis"/"stability.json",orient="records",indent=2)
pd.DataFrame([{"config":r["config"],"peak_gpu_mib":r["peak_gpu_mib"],"peak_temp_c":r["peak_temp_c"],"min_sm_clock_mhz":r["min_sm_clock_mhz"],"mean_gpu_util_pct":r["mean_gpu_util_pct"],"pstates":json.dumps(r["pstates"])} for r in rows]).to_csv(WORK_ROOT/"analysis"/"gpu_health.csv",index=False)
pd.DataFrame([{"config":r["config"],"status":r["status"],"preemption_line_count":r.get("preemption_line_count",0),"error":r.get("error"),
               "gpu_reclaim":json.dumps(next((x.get("gpu_reclaim") for x in ANALYSIS_RECORDS if x["config"]==r["config"]),None),default=str)} for r in rows]).to_csv(WORK_ROOT/"analysis"/"runtime_health.csv",index=False)
print(SUMMARY[["config","status","best_point_n","best_agg_gen_tps","mixed_agg_gen_tps","solver_long_agg_gen_tps","stability_cv","prefix_extension_wall_speedup","preemption_line_count","peak_temp_c","numerics_exact"]].to_string(index=False))

# Long anchor gate
ANCHOR={"expected_sec_per_token_per_seq":ANCHOR_EXPECTED_SEC_PER_TOKEN,"expected_agg_tps_at_8":ANCHOR_BATCH_N/ANCHOR_EXPECTED_SEC_PER_TOKEN,
        "anchor_output_tokens":ANCHOR_OUTPUT_TOKENS,"measured_sec_per_token_per_seq":None,"measured_agg_tps_at_8":None,"ratio_speed":None,"pass":None}
if _anchor:
    ap=_phase(_anchor,f"anchor_long@{ANCHOR_BATCH_N}")
    if ap:
        ANCHOR["measured_sec_per_token_per_seq"]=ap.get("sec_per_output_token_per_seq"); ANCHOR["measured_agg_tps_at_8"]=ap.get("agg_gen_tps")
        if ap.get("agg_gen_tps"):
            ANCHOR["ratio_speed"]=ap["agg_gen_tps"]/ANCHOR["expected_agg_tps_at_8"]; lo,hi=ANCHOR_RATIO_BOUNDS; ANCHOR["pass"]=lo<=ANCHOR["ratio_speed"]<=hi
# End-session frozen-engine drift gate: same decode@8 workload before vs after the ladder.
SESSION_DRIFT={"start_tps":None,"end_tps":None,"ratio":None,"bounds":list(SESSION_DRIFT_RATIO_BOUNDS),"pass":None}
if _anchor and _drift:
    _a8=_phase(_anchor,"decode@8"); _z8=_phase(_drift,"decode@8")
    if _a8 and _z8 and _a8.get("agg_gen_tps") and _z8.get("agg_gen_tps"):
        SESSION_DRIFT.update({"start_tps":_a8["agg_gen_tps"],"end_tps":_z8["agg_gen_tps"],"ratio":_z8["agg_gen_tps"]/_a8["agg_gen_tps"]})
        _lo,_hi=SESSION_DRIFT_RATIO_BOUNDS; SESSION_DRIFT["pass"]=_lo<=SESSION_DRIFT["ratio"]<=_hi
COMPARISON_VALID=(bool(ANCHOR.get("pass")) and bool(SESSION_DRIFT.get("pass")) and not GPU_STATE_COMPROMISED) if USE_MODEL else True

all_valid=[r for r in rows if r["status"]=="ok" and r.get("best_agg_gen_tps")]
valid=[r for r in all_valid if not r.get("experimental_no_solver_promotion")]
# v1.2.4: numerics gate is a threshold on greedy prefix agreement vs C0 (exactness is reported, not required:
# compile fusions, custom all-reduce and TP2 partitioning legitimately change low-order bits).
def _numerics_ok(r):
    if r["config"]==ANCHOR_NAME: return True
    fp=r.get("fingerprint_vs_anchor")
    if not fp: return False
    if NUMERICS_REQUIRE_EXACT_GREEDY: return bool(fp.get("exact"))
    return float(fp.get("mean_prefix_agreement") or 0.0) >= NUMERICS_MIN_PREFIX_AGREEMENT
safe=[r for r in valid if r["config"]!="Z0_frozen_q1_repeat" and r.get("phase_integrity_all") is not False and (r.get("request_metrics_available") is not False) and _numerics_ok(r)]
tp4=[r for r in safe if not r["config"].startswith(("T1_","T2_","T3_"))]
tp2=[r for r in safe if r["config"].startswith(("T1_","T2_"))]
pairs=[r for r in valid if r["config"].startswith("T3_")]
# pair safety: the corresponding single TP2 row must have passed the numerics threshold.
_single_safe={r["config"] for r in tp2}
pairs=[r for r in pairs if (("pair_noAR" in r["config"] and "T1_tp2_noAR" in _single_safe) or ("pair_AR" in r["config"] and "T2_tp2_AR" in _single_safe))]
leader=lambda xs: max(xs,key=lambda r:r["best_agg_gen_tps"]) if xs else None
tp4_leader=leader(tp4); tp2_leader=leader(tp2); pair_leader=leader(pairs); single_leader=leader(tp4+tp2); raw_leader=leader(all_valid)
prefix_rows=[r for r in safe if r.get("prefix_extension_wall_speedup") is not None and "prefix" in r["config"]]
prefix_leader=max(prefix_rows,key=lambda r:r.get("prefix_extension_wall_speedup") or 0) if prefix_rows else None
memory_rows=[r for r in safe if r.get("kv_cache_tokens") is not None]; memory_leader=max(memory_rows,key=lambda r:r.get("kv_cache_tokens") or 0) if memory_rows else None

def short(r):
    return None if not r else {"config":r["config"],"agg_tps":r.get("best_agg_gen_tps"),"n":r.get("best_point_n"),"mixed_tps":r.get("mixed_agg_gen_tps"),"stability_cv":r.get("stability_cv"),"numerics_exact":r.get("numerics_exact")}

followup={"tp4_engine":None,"tp2_single":None,"tp2x2_topology":None}
if COMPARISON_VALID:
    followup={"tp4_engine":None if not tp4_leader else tp4_leader["config"],"tp2_single":None if not tp2_leader else tp2_leader["config"],"tp2x2_topology":None if not pair_leader else pair_leader["config"]}
_cache_rows=[r for r in rows if "prefix" in r["config"] and r["status"]=="ok"]
CACHE_VERIFICATION={r["config"]:{"extension_ratio":r.get("prefix_extension_cache_ratio_expected"),"agent_turn_ratio":r.get("agent_turn_cache_ratio_expected"),
                    "verified":bool((r.get("prefix_extension_cache_ratio_expected") or 0)>=CACHE_HIT_VERIFY_MIN_RATIO)} for r in _cache_rows}
VERDICT={"run_id":RUN_ID,"run_profile":RUN_PROFILE,"fake_engine":FAKE_ENGINE,"comparison_valid":COMPARISON_VALID,"gpu_state_compromised":GPU_STATE_COMPROMISED,"gpu_state_compromised_reason":GPU_STATE_COMPROMISED_REASON,"anchor":ANCHOR,"session_drift":SESSION_DRIFT,"cache_verification":CACHE_VERIFICATION,
         "numerics_gate":{"require_exact":NUMERICS_REQUIRE_EXACT_GREEDY,"min_prefix_agreement":NUMERICS_MIN_PREFIX_AGREEMENT,
                          "per_config":{r["config"]:(r.get("fingerprint_vs_anchor") or None) for r in rows}},
         "engineering_target_agg_tps":THROUGHPUT_TARGET_TPS,"target_provenance":THROUGHPUT_TARGET_PROVENANCE,
         "leaders":{"tp4_numerics_safe":short(tp4_leader),"tp2_single_numerics_safe":short(tp2_leader),"tp2x2_safe":short(pair_leader),
                    "single_engine_numerics_safe":short(single_leader),"prefix_extension":None if not prefix_leader else {"config":prefix_leader["config"],"wall_speedup":prefix_leader["prefix_extension_wall_speedup"],"new_suffix_tps":prefix_leader["prefix_extension_new_suffix_tps"]},
                    "memory_capacity":None if not memory_leader else {"config":memory_leader["config"],"kv_cache_tokens":memory_leader["kv_cache_tokens"]},
                    "raw_throughput_including_experimental":short(raw_leader)},
         "next_solver_smoke_candidates":followup,
         "target_met":bool(COMPARISON_VALID and max([r.get("best_agg_gen_tps") or 0 for r in safe+pairs] or [0])>=THROUGHPUT_TARGET_TPS),
         "experimental_rows_excluded_from_promotion":[r["config"] for r in rows if r.get("experimental_no_solver_promotion")],
         "promotion_note":"Throughput probe never auto-promotes. Anchor + numerics-safe candidates must be tested in the real solver; MTP is experimental due vLLM-0.19 thinking-budget interaction."}
write_json(WORK_ROOT/"analysis"/"throughput_verdict.json",VERDICT)

# Pairwise one-variable effects at a COMMON concurrency point (not each config's best point).
by={r["config"]:r for r in rows}
def _fixed_tps(config,n):
    rec=next((x for x in ANALYSIS_RECORDS if x["config"]==config),None)
    if not rec: return None
    p=next((p for p in (rec.get("result") or {}).get("phases",[]) if p.get("phase")==f"decode@{n}"),None)
    return None if not p else p.get("agg_gen_tps")
def _mixed_tps(config):
    r=by.get(config); return None if not r else r.get("mixed_agg_gen_tps")
pairspec=[
    ("I0_q1_stats","C0_frozen_q1","request_stats_instrumentation",16),
    ("C1_graph_noAR","I0_q1_stats","graph_vs_instrumented_eager",16),
    ("C2_graph_AR","C1_graph_noAR","AR_at_TP4",16),
    ("C3_AR_mns64","C3_noAR_mns64","AR_at_mns64",64),
    ("C4_AR_mns128","C4_noAR_mns128","AR_at_mns128",128),
    ("S1_noAR_mns64_mbt64k","C3_noAR_mns64","mbt64k_vs_32k",64),
    ("S2_noAR_mns64_mbt128k","C3_noAR_mns64","mbt128k_vs_32k",64),
    ("S3_noAR_mns64_async","C3_noAR_mns64","async_scheduling",64),
    ("S4_noAR_mns64_dbo","C3_noAR_mns64","DBO",64),
    ("C5_noAR_prefix64","C3_noAR_mns64","prefix_cache_decode_cold",32),
    ("C5_AR_prefix64","C5_noAR_prefix64","AR_with_prefix",32),
    ("C8_noAR_mns128_gpu95","C4_noAR_mns128","gpu_mem_095",128),
    ("X1_noAR_mns64_mtp1","C3_noAR_mns64","MTP1_experimental",64),
    ("T2_tp2_AR","T1_tp2_noAR","AR_at_TP2",64),
]
effects=[]
for treat,ctrl,label,n in pairspec:
    a,b=by.get(treat),by.get(ctrl); at=_fixed_tps(treat,n); bt=_fixed_tps(ctrl,n); am=_mixed_tps(treat); bm=_mixed_tps(ctrl)
    effects.append({"effect":label,"treatment":treat,"control":ctrl,"common_n":n,
                    "treatment_status":None if not a else a["status"],"control_status":None if not b else b["status"],
                    "fixed_tps_treatment":at,"fixed_tps_control":bt,"fixed_decode_speedup":(at/bt if at and bt else None),
                    "mixed_tps_treatment":am,"mixed_tps_control":bm,"mixed_decode_speedup":(am/bm if am and bm else None),
                    "numerics_exact":None if not a else a.get("numerics_exact")})
pd.DataFrame(effects).to_csv(WORK_ROOT/"analysis"/"pairwise_effects.csv",index=False)
_topology_rows=[]
for r in rows:
    if r["config"].startswith("T3_tp2x2_pair"):
        rec=next((x for x in ANALYSIS_RECORDS if x["config"]==r["config"]),{})
        _topology_rows.append({"config":r["config"],"pairing":"|".join((rec.get("spec") or {}).get("pairing",[])),"status":r["status"],"agg_tps":r.get("best_agg_gen_tps"),"parent_wall_sec":r.get("parent_wall_sec")})
pd.DataFrame(_topology_rows).to_csv(WORK_ROOT/"analysis"/"topology_pairings.csv",index=False)

# Generation-only projections are explicitly secondary.
PROJECTION=[]
proj_leader=max([r for r in (pair_leader,single_leader) if r],key=lambda r:r.get("best_agg_gen_tps") or 0) if (pair_leader or single_leader) else None
if COMPARISON_VALID and proj_leader:
    tps=proj_leader["best_agg_gen_tps"]
    for tasks,label in ((120,"public_eval_120"),(240,"hidden_test_240")):
        for K in (1,2,4,8,16):
            for L in (8192,12288):
                PROJECTION.append({"set":label,"tasks":tasks,"samples_per_task":K,"gen_tokens_per_sample":L,"total_gen_tokens":tasks*K*L,"hours_generation_only":tasks*K*L/tps/3600})
pd.DataFrame(PROJECTION).to_csv(WORK_ROOT/"analysis"/"projection.csv",index=False)

REPORT_PATH=WORK_ROOT/"REPORT.md"
lines=[f"# {EXPERIMENT_ID} — {RUN_PROFILE} — {RUN_ID}","",f"comparison_valid: **{COMPARISON_VALID}** | anchor: `{json.dumps(ANCHOR)}` | drift: `{json.dumps(SESSION_DRIFT)}`","",f"Cache verification: `{json.dumps(CACHE_VERIFICATION)}`","",
       "## Leaders (numerics-safe unless marked raw)","",f"`{json.dumps(VERDICT['leaders'],indent=2)}`","",f"Next solver-smoke candidates: `{json.dumps(followup)}`","",
       "## Measurement files","","- `analysis/summary_table.csv` — per-config aggregate view","- `analysis/phase_metrics.csv` — every physical phase, prompt balance, integrity, energy estimate","- `analysis/common_concurrency.csv` — common n=8/16/32/64 task-ID matched view","- `analysis/pairwise_effects.csv` — one-variable deltas","- `analysis/stability.json` — repeated-wave jitter/CV","- `analysis/gpu_health.csv` — thermal/clock/utilization summary","- `analysis/runtime_health.csv` — preemption/error/reclaim/timeout evidence","- `analysis/topology_pairings.csv` — TP2×2 perfect-matching sweep evidence","- `measurement/batch_plan*.json` — balanced prompt assignment evidence","- `environment/*` — driver/GPU topology/CPU/RAM/disk/pip/runtime evidence","",
       "## Guardrails","","- Real chat-template rendering is fail-closed.","- No solver candidate is emitted if the C0 long anchor fails.","- Single-engine candidates require greedy-fingerprint prefix agreement >= NUMERICS_MIN_PREFIX_AGREEMENT vs C0 (exactness reported separately).","- MTP is measured but excluded from direct solver promotion.","- Raw fixed-length throughput, mixed-length throughput, multi-turn prefix reuse, stability, memory and energy are reported separately.","",
       "## What this does not prove","","Solve quality, final ARC score, leaderboard method attribution, or real multi-turn solver wall-clock. Those require the next real solver smoke."]
REPORT_PATH.write_text("\n".join(lines),encoding="utf-8")
print("ANCHOR:",json.dumps(ANCHOR)); print("VERDICT:",json.dumps({"comparison_valid":COMPARISON_VALID,"leaders":VERDICT["leaders"],"candidates":followup},indent=2)); print("REPORT:",REPORT_PATH)


             config  status  best_point_n  best_agg_gen_tps  mixed_agg_gen_tps  solver_long_agg_gen_tps  stability_cv prefix_extension_wall_speedup  preemption_line_count  peak_temp_c numerics_exact
       C0_frozen_q1      ok          16.0         37.682987          23.586891                      NaN      0.000407                          None                      0         81.0           True
        I0_q1_stats      ok          16.0         37.725467          23.707952                      NaN      0.000206                          None                      0         82.0          False
      C1_graph_noAR      ok          16.0         48.954613          31.448591               129.173172      0.000278                          None                      0         81.0          False
        C2_graph_AR      ok          16.0         48.930432          31.437203               129.042638      0.000069                          None                      0         82.0          False
   C5

In [7]:
# =====================================================================================
# CELL 7 — EVIDENCE BUNDLE / MANIFEST / SELF-CHECK
# =====================================================================================
import zipfile

RUN_METADATA={
    "experiment_id":EXPERIMENT_ID,"run_id":RUN_ID,"run_profile":RUN_PROFILE,"profile":PROFILE,"fake_engine":FAKE_ENGINE,"seed":SEED,
    "q1_frozen_engine":Q1_FROZEN_ENGINE,"q1_frozen_engine_sha256":stable_json_sha256(Q1_FROZEN_ENGINE),"config_ladder":CONFIG_LADDER,
    "model_artifact_fingerprint":MODEL_ARTIFACT_FINGERPRINT_PAYLOAD,"model_artifact_fingerprint_sha256":MODEL_ARTIFACT_FINGERPRINT_SHA256,
    "runtime_stack":RUNTIME_STACK,"runtime_stack_sha256":RUNTIME_STACK_SHA256,"gpu_inventory":GPU_INVENTORY,"prompt_pool_stats":PROMPT_POOL_STATS,
    "worker_sha256":WORKER_SHA256,"parent_source_fingerprint_sha256":PARENT_SOURCE_FINGERPRINT_SHA256,"parent_source_fingerprint_algo":PARENT_SOURCE_FINGERPRINT_ALGO,"measurement_contract":MEASUREMENT_CONTRACT,
    "measurement_constants":{"OUTPUT_TOKENS":OUTPUT_TOKENS,"STABILITY_TOKENS":STABILITY_TOKENS,"STABILITY_REPEATS":STABILITY_REPEATS,"MIXED_BATCH_N":MIXED_BATCH_N,
                             "MIXED_OUTPUT_FRACTIONS":MIXED_OUTPUT_FRACTIONS,"PREFILL_PROMPTS":PREFILL_PROMPTS,"FINGERPRINT_PROMPTS":FINGERPRINT_PROMPTS,
                             "FINGERPRINT_TOKENS":FINGERPRINT_TOKENS,"GPU_SAMPLE_INTERVAL_SEC":GPU_SAMPLE_INTERVAL_SEC,"ANCHOR_OUTPUT_TOKENS":ANCHOR_OUTPUT_TOKENS,"SOLVER_LONG_OUTPUT_TOKENS":SOLVER_LONG_OUTPUT_TOKENS,
                             "TP2_PAIR_N":TP2_PAIR_N,"TOTAL_BUDGET_SEC":TOTAL_BUDGET_SEC},
    "statuses":{r["config"]:r["status"] for r in RECORDS},"topology_status":{r["config"]:r["status"] for r in TOPOLOGY_RECORDS},"session_drift":SESSION_DRIFT,"cache_verification":CACHE_VERIFICATION,
    "comparison_valid":COMPARISON_VALID,"gpu_state_compromised":GPU_STATE_COMPROMISED,"gpu_state_compromised_reason":GPU_STATE_COMPROMISED_REASON,"elapsed_sec":elapsed_sec(),"platform":platform.platform(),"python":platform.python_version(),
}
write_json(WORK_ROOT/"run_metadata.json",RUN_METADATA)
write_json(WORK_ROOT/"measurement"/"PARENT_SOURCE_FINGERPRINT.json",{"algorithm":PARENT_SOURCE_FINGERPRINT_ALGO,"sha256":PARENT_SOURCE_FINGERPRINT_SHA256})

# Explicit handoff + interpretation boundary. These files are included in the runtime bundle so
# the post-run audit knows exactly what evidence was intended and what the probe cannot prove.
RETURN_CONTRACT = {
    "run_id": RUN_ID, "run_profile": RUN_PROFILE, "bundle_name": BUNDLE_NAME,
    "upload_to_chatgpt": [BUNDLE_NAME, "executed_notebook.ipynb"],
    "note": "The runtime evidence bundle contains all generated metrics/logs/manifests; also upload the executed notebook so source-vs-run provenance can be checked."
}
write_json(WORK_ROOT/"RETURN_TO_CHATGPT.json", RETURN_CONTRACT)
KNOWN_LIMITATIONS = {
    "not_measured": [
        "ARC solve quality or exact-grid accuracy",
        "streaming TTFT/ITL as seen by an online API client",
        "full multi-turn tool/verifier wall-clock",
        "leaderboard team/model attribution"
    ],
    "requires_followup_solver_smoke": [
        "any engine/topology candidate from this throughput probe",
        "MTP/speculative decode, especially with thinking-budget control",
        "async scheduling/DBO compatibility with the real agent loop"
    ],
    "interpretation": "Throughput candidates are measurements, not solver promotions. C0 anchor and greedy-numerics gates must pass before comparison is trusted."
}
write_json(WORK_ROOT/"measurement"/"KNOWN_LIMITATIONS.json", KNOWN_LIMITATIONS)

# Manifest is written after SELFCHECK so the selfcheck itself is covered by SHA256 evidence.
MANIFEST={"run_id":RUN_ID,"files":[]}

expected={c["name"] for c in CONFIG_LADDER}; have={p.name[:-len(".record.json")] for p in (WORK_ROOT/"results").glob("*.record.json")}
SELFCHECK={"all_configs_have_record":expected<=have,"missing_records":sorted(expected-have),"statuses":RUN_METADATA["statuses"],
           "comparison_valid":COMPARISON_VALID,"anchor":ANCHOR,"leaders":VERDICT.get("leaders"),"report_present":REPORT_PATH.exists(),
           "required_measurement_files":{str(p.relative_to(WORK_ROOT)):p.exists() for p in [WORK_ROOT/"analysis"/"summary_table.csv",WORK_ROOT/"analysis"/"phase_metrics.csv",WORK_ROOT/"analysis"/"pairwise_effects.csv",WORK_ROOT/"analysis"/"gpu_health.csv",WORK_ROOT/"analysis"/"runtime_health.csv",WORK_ROOT/"analysis"/"topology_pairings.csv",WORK_ROOT/"analysis"/"common_concurrency.csv",WORK_ROOT/"measurement"/"batch_plan.json",WORK_ROOT/"measurement"/"tp2_pair_prompt_plan.json",WORK_ROOT/"measurement"/"PARENT_SOURCE_FINGERPRINT.json",WORK_ROOT/"measurement"/"MEASUREMENT_CONTRACT.json",WORK_ROOT/"measurement"/"KNOWN_LIMITATIONS.json",WORK_ROOT/"RETURN_TO_CHATGPT.json"]}}
SELFCHECK["evidence_integrity_pass"] = bool(SELFCHECK["all_configs_have_record"] and SELFCHECK["report_present"] and all(SELFCHECK["required_measurement_files"].values()))
SELFCHECK["scientific_comparison_valid"] = bool(COMPARISON_VALID)
if FAKE_ENGINE:
    sts=RUN_METADATA["statuses"]
    okrecs=[r for r in RECORDS if r["status"]=="ok"]
    SELFCHECK["selftest_expectations"]={
        "fake_fail":sts.get("ZZ_fake_fail")=="failed","fake_hang":sts.get("ZZ_fake_hang")=="timeout","fake_skip":sts.get("ZZ_fake_skip")=="skipped_deadline",
        "ladder_ok":all(v=="ok" for k,v in sts.items() if not k.startswith("ZZ_")),
        "mixed_phase":all(any(str(p.get("phase","")).startswith("mixed_decode@") for p in (r.get("result") or {}).get("phases",[])) for r in okrecs if not (r.get("spec") or {}).get("drift_sentinel")),
        "prefill_phases_all_rows":all({"prefill_cold","prefill_warm_exact"}<={p.get("phase") for p in (r.get("result") or {}).get("phases",[])} for r in okrecs if not (r.get("spec") or {}).get("drift_sentinel")),
        "extension_phases_prefix_rows":all(any(str(p.get("phase"," ")).startswith("cache_ab_uncached_medium_r") for p in (r.get("result") or {}).get("phases",[])) and any(str(p.get("phase"," ")).startswith("cache_ab_cached_medium_r") for p in (r.get("result") or {}).get("phases",[])) and {"prefill_warm_extension_small","prefill_warm_extension_large"}<={p.get("phase") for p in (r.get("result") or {}).get("phases",[])} for r in okrecs if (r.get("spec") or {}).get("engine_kwargs",{}).get("enable_prefix_caching")),
        "extension_phases_skipped_on_nonprefix_rows":all(any(p.get("phase")=="prefill_extensions" and p.get("skipped") for p in (r.get("result") or {}).get("phases",[])) for r in okrecs if not (r.get("spec") or {}).get("engine_kwargs",{}).get("enable_prefix_caching") and not (r.get("spec") or {}).get("drift_sentinel")),
        "cache_metric_path":all(any(p.get("num_cached_tokens_total") is not None for p in (r.get("result") or {}).get("phases",[])) for r in okrecs if not (r.get("spec") or {}).get("drift_sentinel")),
        "sentinel_ran_last":bool(RECORDS) and (RECORDS[-1].get("spec") or {}).get("drift_sentinel") is True,
        "topology_sweep_path_exercised":sum(1 for r in TOPOLOGY_RECORDS if r["config"].endswith(("p02_13","p03_12")))==2,
        "numerics_gate_threshold_applied":(VERDICT.get("numerics_gate") or {}).get("require_exact") is False,
        "request_metric_path":all(any(p.get("request_timing_available_count",0)>0 for p in (r.get("result") or {}).get("phases",[])) for r in okrecs if (r.get("spec") or {}).get("require_request_metrics")),
        "phase_integrity":all(all(bool(p.get("phase_integrity",True)) for p in (r.get("result") or {}).get("phases",[]) if not p.get("skipped")) for r in okrecs),
        "matched_cache_ab":all(sum(str(p.get("phase"," ")).startswith("cache_ab_uncached_medium_r") for p in (r.get("result") or {}).get("phases",[]))>=CACHE_AB_REPEATS and sum(str(p.get("phase"," ")).startswith("cache_ab_cached_medium_r") for p in (r.get("result") or {}).get("phases",[]))>=CACHE_AB_REPEATS for r in okrecs if (r.get("spec") or {}).get("engine_kwargs",{}).get("enable_prefix_caching")),
        "pair_prompt_disjoint":all(((r.get("result") or {}).get("phases") or [{}])[0].get("replica_prompt_overlap_count")==0 for r in TOPOLOGY_RECORDS if r.get("status")=="ok"),
        "solver_long_phase":any(str(p.get("phase","")).startswith("solver_long@") for r in okrecs for p in (r.get("result") or {}).get("phases",[])),
        "stability_nonprefix":all(any(str(p.get("phase","")).startswith("stability@") for p in (r.get("result") or {}).get("phases",[])) for r in okrecs if not (r.get("spec") or {}).get("engine_kwargs",{}).get("enable_prefix_caching") and not (r.get("spec") or {}).get("drift_sentinel")),
        "both_tp2x2_paths":any("pair_noAR" in r["config"] for r in TOPOLOGY_RECORDS) and any("pair_AR" in r["config"] for r in TOPOLOGY_RECORDS) and all(r["status"]=="ok" for r in TOPOLOGY_RECORDS),
        "drift_sentinel":SESSION_DRIFT.get("ratio") is not None,
        "all_measurement_files":all(SELFCHECK["required_measurement_files"].values()),
    }
    SELFCHECK["selftest_pass"]=SELFCHECK["all_configs_have_record"] and all(SELFCHECK["selftest_expectations"].values())
write_json(WORK_ROOT/"SELFCHECK.json",SELFCHECK)
# Hash every generated evidence file except MANIFEST itself. SELFCHECK is already frozen above.
files=sorted(p for p in WORK_ROOT.rglob("*") if p.is_file() and p.name!="MANIFEST.json")
MANIFEST={"run_id":RUN_ID,"files":[{"path":str(p.relative_to(WORK_ROOT)).replace("\\","/"),"bytes":p.stat().st_size,"sha256":small_file_sha256(p)} for p in files]}
write_json(WORK_ROOT/"MANIFEST.json",MANIFEST)
print("Manifest covers",len(MANIFEST["files"]),"evidence files; manifest sha256",small_file_sha256(WORK_ROOT/"MANIFEST.json"))

BUNDLE_PATH=WORKING_ROOT/BUNDLE_NAME
if BUNDLE_PATH.exists(): BUNDLE_PATH.unlink()
with zipfile.ZipFile(BUNDLE_PATH,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(WORK_ROOT.rglob("*")):
        if p.is_file(): zf.write(p,arcname=str(p.relative_to(WORK_ROOT)).replace("\\","/"))
BUNDLE_SHA256 = small_file_sha256(BUNDLE_PATH)
print("Bundle:",BUNDLE_PATH,f"({BUNDLE_PATH.stat().st_size/1024:.0f} KB, {len(MANIFEST['files'])} manifest files)")
print("Bundle SHA256:", BUNDLE_SHA256)
print("SELFCHECK:",json.dumps(SELFCHECK,indent=2))
if FAKE_ENGINE and not SELFCHECK.get("selftest_pass"):
    raise RuntimeError("SELFTEST_CPU failed; see SELFCHECK above")


Manifest covers 77 evidence files; manifest sha256 e6512dff687f0a1c096f481a30c9d96bb3d26ab97c6da83a3ef9549acec9cce3
Bundle: /kaggle/working/TP_Q27_PROBE_V124_SMOKE_MASTER_EVIDENCE_20260916T091344Z.zip (636 KB, 77 manifest files)
Bundle SHA256: c34fd3736dd9d2565ef681c6fd1936bb7eb1f99b3a95d222eda419049cd3e2aa
SELFCHECK: {
  "all_configs_have_record": true,
  "missing_records": [],
  "statuses": {
    "C0_frozen_q1": "ok",
    "I0_q1_stats": "ok",
    "C1_graph_noAR": "ok",
    "C2_graph_AR": "ok",
    "C5_noAR_prefix64": "timeout",
    "T1_tp2_noAR": "timeout",
    "T2_tp2_AR": "failed",
    "Z0_frozen_q1_repeat": "ok"
  },
  "comparison_valid": true,
  "anchor": {
    "expected_sec_per_token_per_seq": 0.16,
    "expected_agg_tps_at_8": 50.0,
    "anchor_output_tokens": 4096,
    "measured_sec_per_token_per_seq": 0.16562922038891614,
    "measured_agg_tps_at_8": 48.300656014772606,
    "ratio_speed": 0.9660131202954522,
    "pass": true
  },
  "leaders": {
    "tp4_numerics_safe": {
    

## Run order — v1.2.4 MASTER

1. **Run `SMOKE_GPU` first** on Kaggle 4×L4, Internet Off. It is the default. Do not start with `PROBE_GPU`.
2. When the run finishes, upload exactly:
   - `/kaggle/working/TP_Q27_PROBE_V124_SMOKE_MASTER_EVIDENCE_<RUN_ID>.zip`
   - the **executed/output-included notebook**.
3. The MASTER EVIDENCE ZIP already contains raw config specs/results/logs, prompt/tool-history workload snapshots, 2-second GPU telemetry, environment/topology evidence, cache-hit/request-latency metrics, drift sentinel, TP2×2 results, flat analysis CSV/JSON files, `SELFCHECK.json`, and SHA256 manifest.
4. Only after the SMOKE bundle is audited should you switch to `RUN_PROFILE="PROBE_GPU"`.

### Do not interpret a fast row in isolation

A candidate is only comparable when the Q1 long anchor passes **and** the end-of-run frozen-Q1 drift sentinel (now truly last, after the TP2×2 tests) is inside its pre-registered band. Numerics: candidates need greedy prefix agreement ≥ 0.50 vs C0; bit-exactness is reported, not required. Cache rows must show actual `num_cached_tokens`, not only a wall-clock speedup. Request-level timing, preemption, thermal/clock/power telemetry, numerical fingerprint, mixed-length tail behavior, and topology are all separate report dimensions. Any final engine/topology candidate must still pass a real solver SMOKE because this probe does not measure ARC solve quality.
